# Gujarati NLP Mini Project

### Step 1: Preprocessing

In [1]:
%pip install -q indic-nlp-library tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import re
import sys
import unicodedata
from collections import Counter
from functools import lru_cache

import pandas as pd
from tqdm import tqdm
from indicnlp.tokenize import indic_tokenize
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

tqdm.pandas()
pd.set_option('display.max_colwidth', 200)

In [2]:
DATA_PATH = 'nlp_preprocessed_dataset.csv'

df = pd.read_csv(DATA_PATH, usecols=['id', 'headline', 'text'])
df['text'] = df['text'].fillna('').astype(str)
print('Shape:', df.shape)
df.head()

Shape: (50000, 3)


,id,headline,text
0,n338153624,બાંટવા ગામે રવિવારે જુનાગઢના નામાંકિત ડોકટરો દ્વારા સર્વરોગ નિદાન કેમ્પ યોજાશે,"બાંટવા, તા. 3બાંટવા ગામે આગામી તા.05/12/2021 રવિવારના રોજ સમય સવારે 9:00 થી 1:00 શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં.7માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણ..."
1,n407137262,ગાંધીનગરમાં વરસાદી માહોલ વચ્ચે મચ્છરજન્ય રોગચાળો વકર્યો,ગાંધીનગર જિલ્લામાં વરસાદી માહોલની વચ્ચે મચ્છરજન્ય રોગચાળો ધીમે ધીમે માથું ઊંચકી રહ્યો છે. સતત રહેતાં વાદળછાયા વાતાવરણ તેમજ ઠેર ઠેર વરસાદી પાણી ભરાઈ રહેવાના કારણે સાદો તાવ મેલેરિયા અને ડેન્ગ્યુ જે...
2,n394530274,WWEની મહિલા સુપરસ્ટાર રેસલરે લીધી નિવૃત્તિ,ભારત સહિત આખી દુનિયામાં WWE જોનારનો મોટો વર્ગ છે. પરંતુ હાલ WWEના ચાહકો માટે નિરાશાજનક ખબર સામે આવ્યા છે. બે વખતની WWE ચેમ્પિયન પેજે નિવૃતીની જાહેરાત કરી છે. પેજ 7 જુલાઈએ કરાર સમાપ્ત થશે ત્યારે W...
3,n388799324,"હવે આસામમાં બુલડોઝરની એન્ટ્રી, પોલીસ મથકને આગ લગાવનારા પાંચના ઘર તોડી પડાયા","નવી દિલ્હી,તા. 22 મે 2022 હિંસા પર ઉતરી આવતા લોકોના ઘર પર બુલડોઝર ચલાવવાની નવી પ્રથાની યુપી અને એમપી બાદ હવે આસામમાં પણ એન્ટ્રી થઈ છે.આસામના નગાંવ જિલ્લામાં પોલીસે માછલીના વેપારી સફીકુલ ઈસ્લામને અ..."
4,n404948806,16 વર્ષીય બ્રિટિશ પાયલટનું પ્લેન વરસાદમાં ઈંધણ ભરવા માટે અમદાવાદ એરપોર્ટ પર ઉતર્યું,"દુનિયામાં આવી અનેક ઘટનાઓ છે જે ગિનીસ બુક ઓફ વર્લ્ડ રેકોર્ડમાં નોંધાયેલી છે. પરંતુ આ વખતે એક ઘટના સામે આવી છે જે અલગ છે. માત્ર 16 વર્ષની ઉંમરે, બ્રિટીશ-બેલ્જિયન પાઇલટ મેક રધરફોર્ડ પ્લેનમાં એકલા પૃ..."


In [4]:
# Build translation tables once, covering every script:
#   Nd/No -> digits & numerals (0-9, ૦-૯, ½, ², ...)
#   P*/S* -> punctuation & symbols (. , ! ? । ૰ ₹ “ ” emojis, ...)
DIGIT_TABLE, PUNCT_TABLE = {}, {}
for cp in range(sys.maxunicode + 1):
    cat = unicodedata.category(chr(cp))
    if cat in ('Nd', 'No'):
        DIGIT_TABLE[cp] = ' '
    elif cat[0] in ('P', 'S'):
        PUNCT_TABLE[cp] = ' '

ZERO_WIDTH_RE = re.compile('[\u200b-\u200d\u2060\ufeff\u00ad]')
WHITESPACE_RE = re.compile(r'\s+')


def remove_digits(text):
    return text.translate(DIGIT_TABLE)


def remove_punctuation(text):
    return text.translate(PUNCT_TABLE)


def to_lowercase(text):
    # Gujarati script has no case; this only affects embedded Latin text (e.g. "IPL", "Covid").
    return text.lower()


def strip_extra_spaces(text):
    text = ZERO_WIDTH_RE.sub('', text)
    return WHITESPACE_RE.sub(' ', text).strip()


def clean_text(text):
    text = unicodedata.normalize('NFC', text)
    text = remove_digits(text)
    text = remove_punctuation(text)
    text = to_lowercase(text)
    return strip_extra_spaces(text)

In [5]:
# Why not a \w-based regex? It strips Gujarati matras.
w = 'બાંટવા ગામે રવિવારે, તા.05/12'
print('regex [^\\w\\s] :', re.sub(r'[^\w\s]', '', w))
print('clean_text()    :', clean_text(w))

regex [^\w\s] : બટવ ગમ રવવર ત0512
clean_text()    : બાંટવા ગામે રવિવારે તા


In [6]:
# Step-by-step on one article
s = unicodedata.normalize('NFC', df.loc[0, 'text'][:300])
print('RAW               :', s, '\n')
s = remove_digits(s)
print('Digits removed    :', s, '\n')
s = remove_punctuation(s)
print('Punct removed     :', s, '\n')
s = to_lowercase(s)
print('Lowercased        :', s, '\n')
s = strip_extra_spaces(s)
print('Spaces stripped   :', s)

RAW               :  બાંટવા, તા. 3બાંટવા ગામે આગામી તા.05/12/2021 રવિવારના રોજ સમય સવારે 9:00 થી 1:00 શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં.7માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા તેમજ શ્રી લાડી લોહાણા મહિલા મંડળ બાંટવા સહયોગથી જૂનાગઢના નામાંકીત ડોક્ટરો સર્ 

Digits removed    :  બાંટવા, તા.  બાંટવા ગામે આગામી તા.  /  /     રવિવારના રોજ સમય સવારે  :   થી  :   શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં. માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા તેમજ શ્રી લાડી લોહાણા મહિલા મંડળ બાંટવા સહયોગથી જૂનાગઢના નામાંકીત ડોક્ટરો સર્ 

Punct removed     :  બાંટવા  તા   બાંટવા ગામે આગામી તા            રવિવારના રોજ સમય સવારે      થી      શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં  માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા તેમજ શ્રી લાડી લોહાણા મહિલા મંડળ બાંટવા સહયોગથી જૂનાગઢના નામાંકીત ડોક્ટરો સર્ 

Lowercased        :  બાંટવા  તા

In [7]:
df['clean_text'] = df['text'].progress_apply(clean_text)

empty = (df['clean_text'].str.len() == 0).sum()
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)
print(f'Dropped {empty} articles that were empty after cleaning; {len(df)} remain.')
df[['text', 'clean_text']].head(3)

100%|██████████| 50000/50000 [00:14<00:00, 3443.97it/s]


Dropped 0 articles that were empty after cleaning; 50000 remain.


,text,clean_text
0,"બાંટવા, તા. 3બાંટવા ગામે આગામી તા.05/12/2021 રવિવારના રોજ સમય સવારે 9:00 થી 1:00 શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં.7માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણ...",બાંટવા તા બાંટવા ગામે આગામી તા રવિવારના રોજ સમય સવારે થી શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા...
1,ગાંધીનગર જિલ્લામાં વરસાદી માહોલની વચ્ચે મચ્છરજન્ય રોગચાળો ધીમે ધીમે માથું ઊંચકી રહ્યો છે. સતત રહેતાં વાદળછાયા વાતાવરણ તેમજ ઠેર ઠેર વરસાદી પાણી ભરાઈ રહેવાના કારણે સાદો તાવ મેલેરિયા અને ડેન્ગ્યુ જે...,ગાંધીનગર જિલ્લામાં વરસાદી માહોલની વચ્ચે મચ્છરજન્ય રોગચાળો ધીમે ધીમે માથું ઊંચકી રહ્યો છે સતત રહેતાં વાદળછાયા વાતાવરણ તેમજ ઠેર ઠેર વરસાદી પાણી ભરાઈ રહેવાના કારણે સાદો તાવ મેલેરિયા અને ડેન્ગ્યુ જેવા...
2,ભારત સહિત આખી દુનિયામાં WWE જોનારનો મોટો વર્ગ છે. પરંતુ હાલ WWEના ચાહકો માટે નિરાશાજનક ખબર સામે આવ્યા છે. બે વખતની WWE ચેમ્પિયન પેજે નિવૃતીની જાહેરાત કરી છે. પેજ 7 જુલાઈએ કરાર સમાપ્ત થશે ત્યારે W...,ભારત સહિત આખી દુનિયામાં wwe જોનારનો મોટો વર્ગ છે પરંતુ હાલ wweના ચાહકો માટે નિરાશાજનક ખબર સામે આવ્યા છે બે વખતની wwe ચેમ્પિયન પેજે નિવૃતીની જાહેરાત કરી છે પેજ જુલાઈએ કરાર સમાપ્ત થશે ત્યારે wwe છોડ...


In [8]:
GUJARATI_STOPWORDS = '''
હું મેં મને મારું મારા મારી મારો અમે અમને અમારું અમારા અમારી અમારો
આપણે આપણને આપણું આપણા આપણી આપણો તું તને તારું તારા તારી તારો
તમે તમને તમારું તમારા તમારી તમારો
તે તેણે તેને તેનું તેના તેની તેનો તેમાં તેઓ તેઓએ તેમને તેમનું તેમના તેમની તેમનો તેમણે
આ એ આને એને એનું એના એની એનો આમાં એમાં આના આની આનું
જે જેને જેનું જેના જેની જેનો જેમાં જેણે જેઓ
કોઈ કોણ કોને કંઈ કશું શું બધા બધાં બધું બધી દરેક પોતે પોતાના પોતાની પોતાનું પોતાનો
ના ની નું નો ને થી માં પર પરથી માટે સાથે વિશે વિષે દ્વારા સુધી પાસે પાસેથી તરફ તરફથી
બાદ પછી પહેલા પહેલાં વચ્ચે અંદર બહાર ઉપર નીચે સામે
જેવું જેવા જેવી જેવો જેમ તેમ તેમજ વગેરે
જ પણ તો કે અને અથવા પરંતુ છતાં કારણ કેમકે જો ત્યારે જ્યારે ત્યાં જ્યાં અહીં
હવે હજુ હજી ફક્ત માત્ર ખૂબ બહુ વધુ વધારે ઘણા ઘણી ઘણું એટલે એટલું એવું એવા એવી
કેમ ક્યાં ક્યારે કેવી કેવું કેટલા કેટલું નહીં નહિ નહી નથી હા એક
છે છો છું છીએ હતો હતી હતું હતા હતાં હશે હોય હોવા હોવાથી
થયો થયું થઈ થઇ થયા થયેલ થયેલા થાય થશે થવા થતા થતાં
કર્યો કર્યું કરી કર્યા કરે કરવા કરવામાં કરવાની કરશે કરતા કરતાં કરીને કરાયો કરાયું કરાઈ કરાઇ
રહ્યો રહ્યું રહ્યા રહી રહે રહેલ
આવ્યો આવ્યું આવી આવ્યા આવે આવશે આવેલ આવેલા ગયો ગયું ગઈ ગઇ ગયા
'''


def tokenize(text):
    return indic_tokenize.trivial_tokenize(text, lang='gu')


normalizer = IndicNormalizerFactory().get_normalizer('gu')


@lru_cache(maxsize=None)
def normalize_token(token):
    return normalizer.normalize(token)


STOPWORDS = {normalize_token(unicodedata.normalize('NFC', w)) for w in GUJARATI_STOPWORDS.split()}


def remove_stopwords(tokens):
    return [t for t in tokens if t and t not in STOPWORDS]


print('Stopwords:', len(STOPWORDS))

Stopwords: 228


In [9]:
# Step-by-step on one article
tokens = tokenize(df.loc[0, 'clean_text'])
normalized = [normalize_token(t) for t in tokens]
final = remove_stopwords(normalized)
changed = [f'{a} -> {b}' for a, b in zip(tokens, normalized) if a != b]

print('Tokens            :', len(tokens), tokens[:30], '\n')
print('Normalized        :', changed[:10] or 'no tokens changed in this article', '\n')
print('Stopwords removed :', len(normalized) - len(final), sorted(set(normalized) & STOPWORDS), '\n')
print('Final             :', len(final), final[:30])

Tokens            : 114 ['બાંટવા', 'તા', 'બાંટવા', 'ગામે', 'આગામી', 'તા', 'રવિવારના', 'રોજ', 'સમય', 'સવારે', 'થી', 'શ્રી', 'લાડી', 'લોહાણા', 'સિંધી', 'પંચાયતની', 'વાડી', 'તળાવ', 'રોડ', 'ગલી', 'નં', 'માં', 'લાડી', 'લોહાણા', 'સિંધી', 'પંચાયત', 'બાંટવા', 'દ્વારા', 'આયોજિત', 'લાડી'] 

Normalized        : no tokens changed in this article 

Stopwords removed : 9 ['આ', 'છે', 'તેમજ', 'થી', 'દ્વારા', 'પર', 'માં', 'માટે'] 

Final             : 105 ['બાંટવા', 'તા', 'બાંટવા', 'ગામે', 'આગામી', 'તા', 'રવિવારના', 'રોજ', 'સમય', 'સવારે', 'શ્રી', 'લાડી', 'લોહાણા', 'સિંધી', 'પંચાયતની', 'વાડી', 'તળાવ', 'રોડ', 'ગલી', 'નં', 'લાડી', 'લોહાણા', 'સિંધી', 'પંચાયત', 'બાંટવા', 'આયોજિત', 'લાડી', 'લોહાણા', 'સિંધી', 'નવયુવક']


In [10]:
# Full dataset. Token lists are not kept per document (memory); final tokens are stored space-joined.
raw_vocab, norm_vocab, final_vocab = Counter(), Counter(), Counter()
processed, n_tokens, n_final = [], [], []

for text in tqdm(df['clean_text'], desc='Tokenize / normalize / stopwords'):
    tokens = tokenize(text)
    normalized = [normalize_token(t) for t in tokens]
    final = remove_stopwords(normalized)

    raw_vocab.update(tokens)
    norm_vocab.update(normalized)
    final_vocab.update(final)
    processed.append(' '.join(final))
    n_tokens.append(len(tokens))
    n_final.append(len(final))

df['processed_text'] = processed
df['n_tokens'] = n_tokens
df['n_tokens_final'] = n_final
df[['clean_text', 'processed_text']].head(3)

Tokenize / normalize / stopwords: 100%|██████████| 50000/50000 [00:15<00:00, 3206.10it/s]


,clean_text,processed_text
0,બાંટવા તા બાંટવા ગામે આગામી તા રવિવારના રોજ સમય સવારે થી શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા...,બાંટવા તા બાંટવા ગામે આગામી તા રવિવારના રોજ સમય સવારે શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં લાડી લોહાણા સિંધી પંચાયત બાંટવા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા શ્રી લાડી લોહ...
1,ગાંધીનગર જિલ્લામાં વરસાદી માહોલની વચ્ચે મચ્છરજન્ય રોગચાળો ધીમે ધીમે માથું ઊંચકી રહ્યો છે સતત રહેતાં વાદળછાયા વાતાવરણ તેમજ ઠેર ઠેર વરસાદી પાણી ભરાઈ રહેવાના કારણે સાદો તાવ મેલેરિયા અને ડેન્ગ્યુ જેવા...,ગાંધીનગર જિલ્લામાં વરસાદી માહોલની મચ્છરજન્ય રોગચાળો ધીમે ધીમે માથું ઊંચકી સતત રહેતાં વાદળછાયા વાતાવરણ ઠેર ઠેર વરસાદી પાણી ભરાઈ રહેવાના કારણે સાદો તાવ મેલેરિયા ડેન્ગ્યુ રોગોમાં વધારો ગાંધીનગરમાં છે...
2,ભારત સહિત આખી દુનિયામાં wwe જોનારનો મોટો વર્ગ છે પરંતુ હાલ wweના ચાહકો માટે નિરાશાજનક ખબર સામે આવ્યા છે બે વખતની wwe ચેમ્પિયન પેજે નિવૃતીની જાહેરાત કરી છે પેજ જુલાઈએ કરાર સમાપ્ત થશે ત્યારે wwe છોડ...,ભારત સહિત આખી દુનિયામાં wwe જોનારનો મોટો વર્ગ હાલ wweના ચાહકો નિરાશાજનક ખબર બે વખતની wwe ચેમ્પિયન પેજે નિવૃતીની જાહેરાત પેજ જુલાઈએ કરાર સમાપ્ત wwe છોડી દેશે વર્ષીય પેજે ટ્વિટર માહિતી આપી પેજે ટ્વિ...


In [11]:
total_raw, total_final = sum(raw_vocab.values()), sum(final_vocab.values())

stats = pd.Series({
    'Documents': len(df),
    'Total tokens (after tokenization)': total_raw,
    'Total tokens (after stopword removal)': total_final,
    'Stopword tokens removed (%)': round(100 * (1 - total_final / total_raw), 2),
    'Vocabulary size: raw tokens': len(raw_vocab),
    'Vocabulary size: after normalization': len(norm_vocab),
    'Vocabulary size: after stopword removal': len(final_vocab),
    'Avg tokens / doc (before stopwords)': round(df['n_tokens'].mean(), 1),
    'Avg tokens / doc (after stopwords)': round(df['n_tokens_final'].mean(), 1),
})
stats.to_frame('value')

,value
Documents,50000.00
Total tokens (after tokenization),14629680.00
Total tokens (after stopword removal),10140610.00
Stopword tokens removed (%),30.68
Vocabulary size: raw tokens,418260.00
Vocabulary size: after normalization,418260.00
Vocabulary size: after stopword removal,418032.00
Avg tokens / doc (before stopwords),292.60
Avg tokens / doc (after stopwords),202.80


In [12]:
# Most frequent tokens before vs after stopword removal.
# Review the right-hand list: frequent function words still present are candidates to add to GUJARATI_STOPWORDS.
top_before = pd.DataFrame(norm_vocab.most_common(25), columns=['token', 'count'])
top_after = pd.DataFrame(final_vocab.most_common(25), columns=['token', 'count'])
pd.concat({'Before stopword removal': top_before, 'After stopword removal': top_after}, axis=1)

Before stopword removal         After stopword removal       
                     token   count                  token  count
0                       છે  596172                 કહ્યું  31406
1                      અને  266289                    શકે  30721
2                        આ  204336                  કારણે  25771
3                       કે  160787                    મળી  25048
4                       પણ  140694                      ન  24982
5                     માટે  113192                    આજે  24466
6                       પર  108458                   લોકો  24128
7                      કરી  100952                   રીતે  23694
8                      હતી   93151                  પોલીસ  22114
9                        જ   80433                     બે  21886
10                      એક   79076               જણાવ્યું  19930
11                    સાથે   78810                   જોવા  19467
12                     હતો   68049                   તમામ  19355
13                      તે   66715                    આપી  19336
14                     આવી   64736                દરમિયાન  18902
15                     હતા   62703                    શરૂ  17395
16                 કરવામાં   59907                   સૌથી  16663
17                    હતું   56599                    તથા  16334
18                      તો   52629                    કામ  16001
19                  દ્વારા   44050                    વાત  15601
20                     નથી   42434                   વર્ષ  15318
21                      જે   40295                  રહેશે  15255
22                      થઈ   39796                આપવામાં  15094
23                       એ   35808                    કરો  14913
24                     હોય   34888                   અન્ય  14616

In [13]:
OUT_PATH = 'gujarati_exp1_preprocessed.csv'

# utf-8-sig so Excel displays Gujarati correctly
df[['id', 'headline', 'text', 'clean_text', 'processed_text']].to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print('Saved', len(df), 'rows to', OUT_PATH)

Saved 50000 rows to gujarati_exp1_preprocessed.csv


### Step 2: N-Gram Analysis

In [14]:
import math
from indicnlp.tokenize import sentence_tokenize

N_TEST = 1000
MAX_ARTICLES = None   # set e.g. 20000 if RAM runs out while counting trigrams
BOS, EOS, UNK = '<s>', '</s>', '<unk>'


def sentence_tokens(article):
    for sent in sentence_tokenize.sentence_split(article, lang='gu'):
        toks = [normalize_token(t) for t in tokenize(clean_text(sent))]
        toks = [t for t in toks if t]
        if toks:
            yield toks


unigram_counts, bigram_counts, trigram_counts = Counter(), Counter(), Counter()
test_sents = []
n_train_sents = 0

articles = df['text'] if MAX_ARTICLES is None else df['text'].head(MAX_ARTICLES)
for article in tqdm(articles, desc='Counting n-grams'):
    for toks in sentence_tokens(article):
        if len(test_sents) < N_TEST:
            test_sents.append(toks)
            continue
        n_train_sents += 1
        unigram_counts.update(toks)
        unigram_counts[EOS] += 1
        b = [BOS] + toks + [EOS]
        bigram_counts.update(zip(b, b[1:]))
        t = [BOS, BOS] + toks + [EOS]
        trigram_counts.update(zip(t, t[1:], t[2:]))

Counting n-grams: 100%|██████████| 50000/50000 [00:49<00:00, 1008.37it/s]


In [15]:
# History (context) counts: C(w1) for bigrams, C(w1, w2) for trigrams
bigram_hist, trigram_hist = Counter(), Counter()
for (w1, _), c in bigram_counts.items():
    bigram_hist[w1] += c
for (w1, w2, _), c in trigram_counts.items():
    trigram_hist[(w1, w2)] += c

V = len(unigram_counts) + 1   # training words + </s> + <unk>

pd.Series({
    'Training sentences': n_train_sents,
    'Test sentences': len(test_sents),
    'Training tokens': sum(unigram_counts.values()) - unigram_counts[EOS],
    'Vocabulary size V (incl. </s>, <unk>)': V,
    'Unique bigrams': len(bigram_counts),
    'Unique trigrams': len(trigram_counts),
}).to_frame('value')

,value
Training sentences,908263
Test sentences,1000
Training tokens,14554292
"Vocabulary size V (incl. </s>, <unk>)",416588
Unique bigrams,4636913
Unique trigrams,9196745


In [16]:
# Top 10 bigrams / trigrams with MLE conditional probability:
#   P(w2 | w1)     = C(w1, w2)     / C(w1)
#   P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2)
# N-grams containing the sentence-boundary markers <s>, </s> are skipped.
def top_ngrams(counts, hist, n=10):
    rows = []
    for gram, c in counts.most_common(1000):
        if BOS in gram or EOS in gram:
            continue
        h = gram[0] if len(gram) == 2 else gram[:2]
        rows.append({'ngram': ' '.join(gram), 'frequency': c,
                     'history count': hist[h], 'conditional probability': round(c / hist[h], 4)})
        if len(rows) == n:
            break
    return pd.DataFrame(rows, index=range(1, len(rows) + 1))


top_bi = top_ngrams(bigram_counts, bigram_hist)
top_tri = top_ngrams(trigram_counts, trigram_hist)

print('Top 10 bigrams  –  P(w2 | w1)')
display(top_bi)
print('Top 10 trigrams  –  P(w3 | w1, w2)')
display(top_tri)

Top 10 bigrams  –  P(w2 | w1)


,ngram,frequency,history count,conditional probability
1,છે કે,47393,595096,0.0796
2,છે અને,29890,595096,0.0502
3,રહી છે,24847,33370,0.7446
4,શકે છે,24533,30680,0.7996
5,કરવામાં આવી,19689,59683,0.3299
6,રહ્યા છે,19517,30073,0.6490
7,હતું કે,18366,56457,0.3253
8,આવી છે,18048,64538,0.2796
9,આવે છે,18034,25897,0.6964
10,કરી હતી,16974,100687,0.1686


Top 10 trigrams  –  P(w3 | w1, w2)


,ngram,frequency,history count,conditional probability
1,જણાવ્યું હતું કે,9412,11899,0.7910
2,આ પણ વાંચો,7956,9008,0.8832
3,કરવામાં આવી છે,7570,19689,0.3845
4,કરવામાં આવી હતી,6315,19689,0.3207
5,કહ્યું હતું કે,5050,6142,0.8222
6,કરી રહ્યા છે,5014,6869,0.7299
7,થઈ શકે છે,4262,4977,0.8563
8,આવી રહી છે,4173,4617,0.9038
9,કરી શકે છે,3900,5059,0.7709
10,જણાવી દઈએ કે,3897,3924,0.9931


In [17]:
# Evaluate the 1000 held-out sentences.
# Unsmoothed MLE gives P = 0 to any sentence containing an unseen n-gram, so Laplace (add-one) smoothing is used:
#   P(w2 | w1)     = (C(w1, w2) + 1)     / (C(w1) + V)
#   P(w3 | w1, w2) = (C(w1, w2, w3) + 1) / (C(w1, w2) + V)
# Words not seen in training are mapped to <unk>. Probabilities are summed in log10 space to avoid underflow.
def bigram_prob(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (bigram_hist[w1] + V)


def trigram_prob(w1, w2, w3):
    return (trigram_counts[(w1, w2, w3)] + 1) / (trigram_hist[(w1, w2)] + V)


def score_sentence(toks):
    mapped = [w if w in unigram_counts else UNK for w in toks]
    b = [BOS] + mapped + [EOS]
    t = [BOS, BOS] + mapped + [EOS]
    bigrams, trigrams = list(zip(b, b[1:])), list(zip(t, t[1:], t[2:]))
    return {
        'sentence': ' '.join(toks),
        'n_words': len(toks),
        'oov_words': mapped.count(UNK),
        'unseen_bigrams': sum(g not in bigram_counts for g in bigrams),
        'unseen_trigrams': sum(g not in trigram_counts for g in trigrams),
        'log10P_bigram': sum(math.log10(bigram_prob(*g)) for g in bigrams),
        'log10P_trigram': sum(math.log10(trigram_prob(*g)) for g in trigrams),
    }


eval_df = pd.DataFrame([score_sentence(s) for s in test_sents])
n_pred = eval_df['n_words'] + 1   # every word plus </s> is predicted
eval_df['PPL_bigram'] = 10 ** (-eval_df['log10P_bigram'] / n_pred)
eval_df['PPL_trigram'] = 10 ** (-eval_df['log10P_trigram'] / n_pred)

print(f'Test sentences: {len(eval_df)}  |  words: {eval_df["n_words"].sum()}  |  OOV words: {eval_df["oov_words"].sum()}')
for m in ('bigram', 'trigram'):
    corpus_ppl = 10 ** (-eval_df[f'log10P_{m}'].sum() / n_pred.sum())
    zero_mle = (eval_df[f'unseen_{m}s'] > 0).mean() * 100
    print(f'{m:8s}: corpus perplexity = {corpus_ppl:10.1f}  |  '
          f'sentences with P = 0 under unsmoothed MLE: {zero_mle:.1f}%')

eval_df.head(10)

Test sentences: 1000  |  words: 16022  |  OOV words: 352
bigram  : corpus perplexity =    13834.9  |  sentences with P = 0 under unsmoothed MLE: 83.8%
trigram : corpus perplexity =    85171.4  |  sentences with P = 0 under unsmoothed MLE: 90.6%


,sentence,n_words,oov_words,unseen_bigrams,unseen_trigrams,log10P_bigram,log10P_trigram,PPL_bigram,PPL_trigram
0,બાંટવા તા,2,0,1,2,-13.136756,-17.060552,23928.667141,486239.831317
1,બાંટવા ગામે આગામી તા,4,0,1,2,-21.378887,-25.475544,18870.240359,124482.642547
2,રવિવારના રોજ સમય સવારે થી શ્રી લાડી લોહાણા સિંધી પંચાયતની વાડી તળાવ રોડ ગલી નં,15,0,7,11,-76.751196,-85.163262,62654.140202,210234.440231
3,માં લાડી લોહાણા સિંધી પંચાયત બાંટવા દ્વારા આયોજિત લાડી લોહાણા સિંધી નવયુવક મંડળ બાંટવા તેમજ શ્રી લાડી લોહાણા મહિલા મંડળ બાંટવા સહયોગથી જૂનાગઢના નામાંકીત ડોક્ટરો સર્વરોગ નિદાન ફ્રી કેમ્પનું આયોજન ક...,36,0,25,34,-185.843669,-202.560564,105390.594562,298270.189183
4,વિનય હરિયાણી ડો,3,0,1,2,-18.210776,-21.638876,35702.129770,256873.334236
5,મોહિત મારવાણીયા ડો,3,0,2,3,-18.210804,-21.638887,35702.686671,256874.875709
6,હર્ષાબેન ગાધે ડો,3,0,3,4,-19.553098,-22.981287,77316.915051,556316.353006
7,કૃણાલ ભાલોડી ડો,3,0,2,3,-18.138304,-21.566340,34243.330543,246368.400952
8,વિશાલ પટેલ ડો,3,0,0,1,-16.288960,-20.082614,11809.732980,104870.553877
9,પાયલ ભાટુ ડો,3,0,2,3,-17.996999,-21.425021,31568.191463,227119.970241


In [18]:
# Sentence with the highest probability under each model.
# Joint probability always favours short sentences (fewer factors < 1 multiplied together),
# so the best sentence by per-word probability (lowest perplexity, >= 5 words) is shown as well.
def show(title, row, m):
    print(f'{title}')
    print(f'  {row["sentence"]}')
    print(f'  words = {row["n_words"]}  |  log10 P = {row[f"log10P_{m}"]:.3f}  '
          f'(P = {10 ** row[f"log10P_{m}"]:.3e})  |  perplexity = {row[f"PPL_{m}"]:.1f}\n')


long_sents = eval_df[eval_df['n_words'] >= 5]
for m in ('bigram', 'trigram'):
    show(f'[{m}] Highest joint probability:', eval_df.loc[eval_df[f'log10P_{m}'].idxmax()], m)
    show(f'[{m}] Highest per-word probability (>= 5 words):', long_sents.loc[long_sents[f'PPL_{m}'].idxmin()], m)

[bigram] Highest joint probability:
  com
  words = 1  |  log10 P = -5.566  (P = 2.719e-06)  |  perplexity = 606.4

[bigram] Highest per-word probability (>= 5 words):
  પોલીસે તપાસ હાથ ધરી છે
  words = 5  |  log10 P = -12.705  (P = 1.971e-13)  |  perplexity = 131.1

[trigram] Highest joint probability:
  com
  words = 1  |  log10 P = -5.581  (P = 2.626e-06)  |  perplexity = 617.2

[trigram] Highest per-word probability (>= 5 words):
  તમારા ફોન પર લેટેસ્ટ ન્યૂઝ અપડેટ્સ સૌથી પહેલા મેળવવા માટે હમણાં જ sandesh ની મોબાઇલ એપ્લિકેશન ડાઉનલોડ કરો
  words = 18  |  log10 P = -45.737  (P = 1.834e-46)  |  perplexity = 255.4



In [19]:
# Next-word prediction: trigram model on the last two words, backing off to the bigram model on the last word.
# (Scans the n-gram tables, so each call takes a few seconds.)
def predict_next(context, k=5):
    toks = [normalize_token(t) for t in tokenize(clean_text(context))]
    if len(toks) >= 2:
        h = (toks[-2], toks[-1])
        cands = Counter({g[2]: c for g, c in trigram_counts.items() if g[0] == h[0] and g[1] == h[1]})
        if cands:
            return 'trigram', [(w, round(c / trigram_hist[h], 4)) for w, c in cands.most_common(k)]
    if toks:
        h = toks[-1]
        cands = Counter({g[1]: c for g, c in bigram_counts.items() if g[0] == h})
        if cands:
            return 'bigram', [(w, round(c / bigram_hist[h], 4)) for w, c in cands.most_common(k)]
    return None, []


contexts = [
    ' '.join(top_tri.loc[1, 'ngram'].split()[:2]),
    top_bi.loc[1, 'ngram'].split()[0],
    ' '.join(test_sents[1][:3]),
]
for ctx in contexts:
    model, preds = predict_next(ctx)
    print(f'{ctx}  ->  ({model})')
    for w, p in preds:
        print(f'    {w:15s} {p}')

જણાવ્યું હતું  ->  (trigram)
    કે              0.791
    </s>            0.1888
    અને             0.0039
    પરંતુ           0.0014
    જેથી            0.0007
છે  ->  (bigram)
    </s>            0.6807
    કે              0.0796
    અને             0.0502
    જે              0.0147
    ત્યારે          0.0107
બાંટવા ગામે આગામી  ->  (trigram)
    તા              0.5
    મી              0.5


### Step 3: Rule Based POS Tagging

In [20]:
import numpy as np
from collections import defaultdict


class RuleBasedPOSTagger:
    def __init__(self):
        self.noun_suffixes = ['ો', 'ી', 'ું', 'ા', 'આ', 'એ', 'ઓ', 'ાઓ', 'ીઓ', 'માં', 'ને', 'થી']
        self.verb_suffixes = ['ે', 'યો', 'યા', 'યું', 'શે', 'છે', 'ત', 'વું', 'વી', 'વા', 'તા', 'તી', 'ાવું']
        self.adj_suffixes = ['ીય', 'લી', 'વાળો', 'વાળી', 'વાળું', 'દાર', 'માન']
        self.adv_suffixes = ['થી', 'માં', 'એ']

        self.common_nouns = {
            'ગામ', 'માણસ', 'ઘર', 'શહેર', 'કામ', 'દિવસ', 'રાત', 'સવાર', 'સાંજ',
            'ડોક્ટર', 'ડાક્ટર', 'શિક્ષક', 'વિદ્યાર્થી', 'બાળક', 'સ્ત્રી', 'પુરુષ',
            'પાણી', 'અન્ન', 'કેમ્પ', 'નિદાન', 'સેવા', 'યોજના', 'કાર્યક્રમ', 'પંચાયત',
            'રવિવાર', 'સોમવાર', 'મંગળવાર', 'નામ', 'સ્થળ', 'જગ્યા', 'વિસ્તાર'
        }

        self.common_verbs = {
            'કરવું', 'જવું', 'આવવું', 'થવું', 'રહેવું', 'લખવું', 'વાંચવું', 'બોલવું',
            'યોજાશે', 'કરે', 'કરશે', 'થશે', 'જાય', 'આવે', 'રહે', 'છે', 'હતો', 'હતી',
            'કરી', 'કર્યું', 'થયું', 'આવ્યો', 'ગયો', 'રહ્યો'
        }

        self.common_adj = {
            'સારો', 'નવો', 'જૂનો', 'મોટો', 'નાનો', 'સુંદર', 'સ્વચ્છ', 'ખરાબ',
            'નામાંકિત', 'પ્રખ્યાત', 'પ્રસિદ્ધ', 'લોકપ્રિય', 'મહત્વપૂર્ણ', 'જરૂરી'
        }

        self.common_adv = {
            'અહીં', 'ત્યાં', 'આજે', 'કાલે', 'હવે', 'પછી', 'પહેલાં', 'ફરી', 'હંમેશા',
            'ક્યારેક', 'સદા', 'વારંવાર', 'જલ્દી', 'ધીમે'
        }

        self.common_pronouns = {
            'તે', 'આ', 'હું', 'તું', 'અમે', 'તમે', 'તેઓ', 'આપણે', 'કોઇ', 'શું',
            'કોણ', 'ક્યાં', 'કેવી', 'કેવો', 'તારુ', 'મારુ', 'આપણુ'
        }

        self.common_postpositions = {
            'માં', 'ને', 'થી', 'સુધી', 'માટે', 'સાથે', 'વિના', 'દ્વારા', 'પાસે',
            'ઉપર', 'નીચે', 'આગળ', 'પાછળ', 'વચ્ચે', 'પર'
        }

        self.common_conjunctions = {
            'અને', 'કે', 'પણ', 'કારણ', 'જો', 'તો', 'તેથી', 'માટે', 'અથવા', 'કિંવા'
        }

        self.common_particles = {
            'જ', 'તો', 'ને', 'કે', 'શું', 'ના', 'નહિ', 'નહી'
        }

        self.number_pattern = re.compile(r'^\d+$')

        self.punctuation = {'.', ',', '?', '!', ';', ':', '-', '(', ')', '[', ']', '"', "'"}

    def tag_word(self, word, prev_tag=None):
        """Apply rules to tag a single word"""
        if word in self.punctuation:
            return 'PUNCT'

        if self.number_pattern.match(word):
            return 'NUM'

        if word in self.common_pronouns:
            return 'PRON'
        elif word in self.common_postpositions:
            return 'ADP'
        elif word in self.common_conjunctions:
            return 'CCONJ'
        elif word in self.common_particles:
            return 'PART'
        elif word in self.common_verbs:
            return 'VERB'
        elif word in self.common_adj:
            return 'ADJ'
        elif word in self.common_adv:
            return 'ADV'
        elif word in self.common_nouns:
            return 'NOUN'

        for suffix in self.verb_suffixes:
            if word.endswith(suffix) and len(word) > 2:
                return 'VERB'

        for suffix in self.adj_suffixes:
            if word.endswith(suffix) and len(word) > 3:
                return 'ADJ'

        for suffix in self.adv_suffixes:
            if word.endswith(suffix) and len(word) > 2 and prev_tag in ['VERB', 'ADJ']:
                return 'ADV'

        for suffix in self.noun_suffixes:
            if word.endswith(suffix) and len(word) > 2:
                return 'NOUN'

        if word and word[0].isupper():
            return 'PROPN'

        return 'NOUN'

    def tag_sentence(self, tokens):
        """Tag a token list left to right, passing each predicted tag on as prev_tag"""
        tags, prev_tag = [], None
        for word in tokens:
            prev_tag = self.tag_word(word, prev_tag)
            tags.append(prev_tag)
        return list(zip(tokens, tags))


rule_tagger = RuleBasedPOSTagger()
rule_tagger.tag_sentence('બાંટવા ગામે રવિવારે ડોક્ટરો દ્વારા નિદાન કેમ્પ યોજાશે .'.split())

[('બાંટવા', 'VERB'),
 ('ગામે', 'VERB'),
 ('રવિવારે', 'VERB'),
 ('ડોક્ટરો', 'NOUN'),
 ('દ્વારા', 'ADP'),
 ('નિદાન', 'NOUN'),
 ('કેમ્પ', 'NOUN'),
 ('યોજાશે', 'VERB'),
 ('.', 'PUNCT')]

In [21]:
# POS corpus: sentences tokenized from the raw text WITHOUT the Step 1 cleaning, because the rules
# rely on punctuation (PUNCT), digits (NUM) and capital letters (PROPN), which cleaning removes.
# No gold-annotated Gujarati POS corpus is available, so the rule-based tagger labels the corpus
# ("silver" labels); the HMM is trained on those labels.
N_POS_TRAIN, N_POS_TEST = 20000, 2000


def pos_sentence_tokens(article):
    for sent in sentence_tokenize.sentence_split(article, lang='gu'):
        toks = [normalize_token(t) for t in tokenize(strip_extra_spaces(unicodedata.normalize('NFC', sent)))]
        toks = [t for t in toks if t.strip()]
        if toks:
            yield toks


pos_sents = []
for article in df['text']:
    pos_sents.extend(pos_sentence_tokens(article))
    if len(pos_sents) >= N_POS_TRAIN + N_POS_TEST:
        break

silver = [rule_tagger.tag_sentence(s) for s in pos_sents[:N_POS_TRAIN + N_POS_TEST]]
pos_train, pos_test = silver[:N_POS_TRAIN], silver[N_POS_TRAIN:]
print(f'Train sentences: {len(pos_train)} ({sum(map(len, pos_train))} tokens)  |  '
      f'Test sentences: {len(pos_test)} ({sum(map(len, pos_test))} tokens)')

tag_dist = Counter(t for s in pos_train for _, t in s)
pd.DataFrame(tag_dist.most_common(), columns=['tag', 'count']).assign(
    percent=lambda d: (100 * d['count'] / d['count'].sum()).round(2))

Train sentences: 20000 (370464 tokens)  |  Test sentences: 2000 (37690 tokens)


,tag,count,percent
0,NOUN,194119,52.40
1,VERB,82358,22.23
2,PUNCT,38155,10.30
3,CCONJ,14572,3.93
4,ADP,9919,2.68
5,PRON,7963,2.15
6,NUM,7292,1.97
7,ADV,6612,1.78
8,ADJ,3751,1.01
9,PROPN,3251,0.88


In [22]:
# First-order HMM tagger with Viterbi decoding.
#   Transition  P(t_i | t_{i-1}) : add-one smoothed, with a <START> state
#   Emission    P(w | t)         : add-k smoothed
# Unknown words: training words seen only once are replaced by a word-shape class
# (<NUM>, <PUNCT>, <UPPER>, or <SUF-xx> = last two characters), so the model learns
# P(class | tag) and can tag unseen test words by their shape/suffix.
def word_shape(w):
    if w.isdigit():
        return '<NUM>'
    if all(unicodedata.category(ch)[0] in 'PS' for ch in w):
        return '<PUNCT>'
    if w[0].isupper():
        return '<UPPER>'
    return f'<SUF-{w[-2:]}>'


class HMMTagger:
    def __init__(self, k=0.01):
        self.k = k

    def obs(self, w):
        return w if w in self.known else word_shape(w)

    def fit(self, tagged_sents):
        word_freq = Counter(w for s in tagged_sents for w, _ in s)
        self.known = {w for w, c in word_freq.items() if c > 1}
        self.tags = sorted({t for s in tagged_sents for _, t in s})
        self.tag_idx = {t: i for i, t in enumerate(self.tags)}
        n = len(self.tags)

        trans = np.zeros((n + 1, n))                 # last row = <START>
        emit = defaultdict(lambda: np.zeros(n))
        for s in tagged_sents:
            prev = n
            for w, t in s:
                j = self.tag_idx[t]
                trans[prev, j] += 1
                emit[self.obs(w)][j] += 1
                prev = j

        self.trans_counts = trans
        self.log_trans = np.log((trans + 1) / (trans.sum(axis=1, keepdims=True) + n))
        tag_totals = sum(emit.values())
        denom = tag_totals + self.k * len(emit)
        self.log_emit = {o: np.log((c + self.k) / denom) for o, c in emit.items()}
        self.unseen = np.zeros(n)                    # unseen word class: let transitions decide
        return self

    def tag(self, words):
        emissions = [self.log_emit.get(self.obs(w), self.unseen) for w in words]
        score = self.log_trans[-1] + emissions[0]
        backptrs = []
        for e in emissions[1:]:
            cand = score[:, None] + self.log_trans[:-1]   # cand[i, j]: best path ending in i, then i -> j
            backptrs.append(cand.argmax(axis=0))
            score = cand.max(axis=0) + e
        best = [int(score.argmax())]
        for bp in reversed(backptrs):
            best.append(int(bp[best[-1]]))
        return list(zip(words, [self.tags[i] for i in reversed(best)]))


hmm_tagger = HMMTagger().fit(pos_train)
print('Tags:', hmm_tagger.tags)
print('Known words:', len(hmm_tagger.known), '| emission observations (words + shape classes):', len(hmm_tagger.log_emit))

# Transition probability matrix P(column tag | row tag)
pd.DataFrame(np.exp(hmm_tagger.log_trans), index=hmm_tagger.tags + ['<START>'], columns=hmm_tagger.tags).round(3)

Tags: ['ADJ', 'ADP', 'ADV', 'CCONJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'VERB']
Known words: 21070 | emission observations (words + shape classes): 22484


,ADJ,ADP,ADV,CCONJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,VERB
ADJ,0.013,0.012,0.088,0.020,0.547,0.009,0.005,0.007,0.003,0.031,0.265
ADP,0.024,0.007,0.007,0.021,0.662,0.032,0.018,0.030,0.005,0.020,0.175
ADV,0.014,0.008,0.011,0.057,0.517,0.020,0.016,0.035,0.002,0.088,0.232
CCONJ,0.012,0.002,0.015,0.026,0.640,0.018,0.002,0.071,0.006,0.071,0.137
NOUN,0.010,0.036,0.005,0.036,0.572,0.017,0.007,0.009,0.004,0.071,0.234
NUM,0.012,0.048,0.002,0.014,0.662,0.009,0.008,0.001,0.009,0.160,0.077
PART,0.028,0.000,0.003,0.011,0.660,0.009,0.008,0.020,0.015,0.018,0.228
PRON,0.012,0.031,0.021,0.037,0.606,0.008,0.016,0.015,0.003,0.005,0.247
PROPN,0.004,0.026,0.002,0.014,0.338,0.033,0.004,0.003,0.274,0.272,0.031
PUNCT,0.009,0.015,0.006,0.027,0.625,0.071,0.006,0.032,0.041,0.071,0.097


In [23]:
def evaluate(reference_sents, predicted_sents):
    ref = [t for s in reference_sents for _, t in s]
    pred = [t for s in predicted_sents for _, t in s]
    acc = np.mean([r == p for r, p in zip(ref, pred)])
    rows = []
    for tag in sorted(set(ref) | set(pred)):
        tp = sum(r == p == tag for r, p in zip(ref, pred))
        n_pred, n_ref = pred.count(tag), ref.count(tag)
        prec = tp / n_pred if n_pred else 0.0
        rec = tp / n_ref if n_ref else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        rows.append({'tag': tag, 'precision': prec, 'recall': rec, 'f1': f1, 'support': n_ref})
    report = pd.DataFrame(rows).set_index('tag').round(3)
    confusion = pd.crosstab(pd.Series(ref, name='reference'), pd.Series(pred, name='predicted'))
    return acc, report, confusion


# HMM on the 2000 held-out sentences, compared with the rule-based labels it was trained to imitate
hmm_test_pred = [hmm_tagger.tag([w for w, _ in s]) for s in tqdm(pos_test, desc='Viterbi')]
acc, report, confusion = evaluate(pos_test, hmm_test_pred)

test_words = [w for s in pos_test for w, _ in s]
oov_rate = np.mean([w not in hmm_tagger.known for w in test_words])
print(f'HMM agreement with rule-based tags on held-out sentences: {acc:.2%}  (unknown-word rate {oov_rate:.2%})')
display(report)
confusion

Viterbi: 100%|██████████| 2000/2000 [00:00<00:00, 11802.22it/s]


HMM agreement with rule-based tags on held-out sentences: 99.17%  (unknown-word rate 12.42%)


,precision,recall,f1,support
tag,,,,
ADJ,0.990,0.990,0.990,381
ADP,1.000,1.000,1.000,1044
ADV,0.981,0.700,0.817,751
CCONJ,1.000,1.000,1.000,1621
NOUN,0.987,0.997,0.992,19358
NUM,1.000,1.000,1.000,658
PART,1.000,1.000,1.000,263
PRON,1.000,1.000,1.000,855
PROPN,0.910,0.986,0.946,358


predicted,ADJ,ADP,ADV,CCONJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,VERB
reference,,,,,,,,,,,
ADJ,377,0,0,0,4,0,0,0,0,0,0
ADP,0,1044,0,0,0,0,0,0,0,0,0
ADV,0,0,526,0,224,0,0,0,1,0,0
CCONJ,0,0,0,1621,0,0,0,0,0,0,0
NOUN,4,0,10,0,19306,0,0,0,32,0,6
NUM,0,0,0,0,0,658,0,0,0,0,0
PART,0,0,0,0,0,0,263,0,0,0,0
PRON,0,0,0,0,0,0,0,855,0,0,0
PROPN,0,0,0,0,5,0,0,0,353,0,0


In [24]:
# Agreement with the rule tagger only shows how well the HMM imitates the rules. To measure real tagging
# accuracy, both taggers are scored on a small hand-annotated gold set (news-style sentences, same tagset).
GOLD = '''
રવિવારે/NOUN ગામમાં/NOUN આરોગ્ય/NOUN કેમ્પ/NOUN યોજાશે/VERB ./PUNCT
મુખ્યમંત્રીએ/NOUN આજે/ADV નવી/ADJ યોજનાની/NOUN જાહેરાત/NOUN કરી/VERB ./PUNCT
પોલીસે/NOUN આરોપીની/NOUN ધરપકડ/NOUN કરી/VERB હતી/VERB ./PUNCT
અમદાવાદમાં/PROPN ભારે/ADJ વરસાદ/NOUN પડ્યો/VERB અને/CCONJ રસ્તાઓ/NOUN પર/ADP પાણી/NOUN ભરાયા/VERB ./PUNCT
તેઓ/PRON દરરોજ/ADV આ/PRON મંદિરની/NOUN મુલાકાત/NOUN લે/VERB છે/VERB ./PUNCT
ભારતીય/ADJ ટીમે/NOUN ફાઇનલ/ADJ મેચમાં/NOUN શાનદાર/ADJ જીત/NOUN મેળવી/VERB ./PUNCT
વિદ્યાર્થીઓ/NOUN માટે/ADP શાળામાં/NOUN મફત/ADJ પુસ્તકો/NOUN આપવામાં/VERB આવશે/VERB ./PUNCT
હું/PRON કાલે/ADV મારા/PRON મિત્ર/NOUN સાથે/ADP સુરત/PROPN જઈશ/VERB ./PUNCT
આ/PRON ઘટનામાં/NOUN બે/NUM લોકોના/NOUN મોત/NOUN થયા/VERB છે/VERB ./PUNCT
સરકારે/NOUN ખેડૂતો/NOUN માટે/ADP ૫૦૦/NUM કરોડ/NUM રૂપિયાની/NOUN સહાય/NOUN જાહેર/ADJ કરી/VERB ./PUNCT
શહેરમાં/NOUN કોરોનાના/NOUN નવા/ADJ કેસ/NOUN નોંધાયા/VERB નથી/VERB ./PUNCT
ડોક્ટરે/NOUN દર્દીને/NOUN તરત/ADV જ/PART હોસ્પિટલ/NOUN ખસેડ્યો/VERB ./PUNCT
પ્રધાનમંત્રી/NOUN નરેન્દ્ર/PROPN મોદીએ/PROPN ગાંધીનગરમાં/PROPN નવા/ADJ પુલનું/NOUN ઉદ્ઘાટન/NOUN કર્યું/VERB ./PUNCT
આ/PRON કાર્યક્રમમાં/NOUN મોટી/ADJ સંખ્યામાં/NOUN લોકો/NOUN હાજર/ADJ રહ્યા/VERB હતા/VERB ./PUNCT
તેમણે/PRON કહ્યું/VERB કે/CCONJ આ/PRON નિર્ણય/NOUN ખૂબ/ADV મહત્વપૂર્ણ/ADJ છે/VERB ./PUNCT
IPL/PROPN માં/ADP ગુજરાત/PROPN ટાઇટન્સે/PROPN શાનદાર/ADJ પ્રદર્શન/NOUN કર્યું/VERB ./PUNCT
ખેડૂતો/NOUN ખેતરમાં/NOUN વહેલી/ADJ સવારે/NOUN કામ/NOUN કરે/VERB છે/VERB ,/PUNCT પરંતુ/CCONJ પાણી/NOUN ઓછું/ADJ છે/VERB ./PUNCT
શું/PART તમે/PRON આ/PRON સમાચાર/NOUN વાંચ્યા/VERB ?/PUNCT
શાળાના/NOUN ૧૨/NUM વિદ્યાર્થીઓને/NOUN ઇનામ/NOUN આપવામાં/VERB આવ્યું/VERB ./PUNCT
વડોદરા/PROPN શહેરમાં/NOUN નવી/ADJ બસ/NOUN સેવા/NOUN આવતીકાલથી/ADV મળશે/VERB ./PUNCT
'''

gold_sents = [
    [(normalize_token(unicodedata.normalize('NFC', w)), t) for w, t in (tok.rsplit('/', 1) for tok in line.split())]
    for line in GOLD.strip().splitlines()
]
rule_gold = [rule_tagger.tag_sentence([w for w, _ in s]) for s in gold_sents]
hmm_gold = [hmm_tagger.tag([w for w, _ in s]) for s in gold_sents]

rule_acc, rule_rep, _ = evaluate(gold_sents, rule_gold)
hmm_acc, hmm_rep, _ = evaluate(gold_sents, hmm_gold)
print(f'Gold set: {len(gold_sents)} sentences, {sum(map(len, gold_sents))} tokens')
print(f'Rule-based accuracy: {rule_acc:.2%}')
print(f'HMM accuracy       : {hmm_acc:.2%}')

pd.concat({'Rule-based': rule_rep[['precision', 'recall', 'f1']],
           'HMM': hmm_rep[['precision', 'recall', 'f1']],
           'Gold': rule_rep[['support']]}, axis=1)

Gold set: 20 sentences, 162 tokens
Rule-based accuracy: 70.99%
HMM accuracy       : 70.99%


Rule-based                     HMM                  Gold
       precision recall     f1 precision recall     f1 support
tag                                                           
ADJ        1.000  0.312  0.476     1.000  0.312  0.476      16
ADP        1.000  1.000  1.000     1.000  1.000  1.000       5
ADV        0.500  0.333  0.400     0.500  0.333  0.400       6
CCONJ      1.000  0.667  0.800     1.000  0.667  0.800       3
NOUN       0.667  0.764  0.712     0.667  0.764  0.712      55
NUM        1.000  0.500  0.667     1.000  0.500  0.667       4
PART       1.000  0.500  0.667     1.000  0.500  0.667       2
PRON       0.889  0.800  0.842     0.889  0.800  0.842      10
PROPN      1.000  0.111  0.200     1.000  0.111  0.200       9
PUNCT      1.000  1.000  1.000     1.000  1.000  1.000      21
VERB       0.531  0.839  0.650     0.531  0.839  0.650      31

In [ ]:
# Word-by-word comparison on the gold set (X marks a wrong tag)
rows = []
for gs, rs, hs in zip(gold_sents, rule_gold, hmm_gold):
    for (w, g), (_, r), (_, h) in zip(gs, rs, hs):
        rows.append({'word': w, 'gold': g,
                     'rule-based': r + ('' if r == g else ' X'),
                     'HMM': h + ('' if h == g else ' X')})
comparison = pd.DataFrame(rows)

# Show the first few sentences in full, then only the tokens where at least one tagger is wrong
print('Sentence 1-3:')
display(comparison.head(sum(map(len, gold_sents[:3]))))
print('All tokens where at least one tagger disagrees with gold:')
comparison[comparison['rule-based'].str.endswith('X') | comparison['HMM'].str.endswith('X')]

Sentence 1-3:


,word,gold,rule-based,HMM
0,રવિવારે,NOUN,VERB ✗,VERB ✗
1,ગામમાં,NOUN,ADV ✗,ADV ✗
2,આરોગ્ય,NOUN,NOUN,NOUN
3,કેમ્પ,NOUN,NOUN,NOUN
4,યોજાશે,VERB,VERB,VERB
5,.,PUNCT,PUNCT,PUNCT
6,મુખ્યમંત્રીએ,NOUN,NOUN,NOUN
7,આજે,ADV,ADV,ADV
8,નવી,ADJ,VERB ✗,VERB ✗
9,યોજનાની,NOUN,NOUN,NOUN


All tokens where at least one tagger disagrees with gold:


,word,gold,rule-based,HMM
0,રવિવારે,NOUN,VERB ✗,VERB ✗
1,ગામમાં,NOUN,ADV ✗,ADV ✗
8,નવી,ADJ,VERB ✗,VERB ✗
10,જાહેરાત,NOUN,VERB ✗,VERB ✗
13,પોલીસે,NOUN,VERB ✗,VERB ✗
19,અમદાવાદમાં,PROPN,NOUN ✗,NOUN ✗
20,ભારે,ADJ,VERB ✗,VERB ✗
30,દરરોજ,ADV,NOUN ✗,NOUN ✗
33,મુલાકાત,NOUN,VERB ✗,VERB ✗
34,લે,VERB,NOUN ✗,NOUN ✗


### Step 4: CFG Parsing

In [37]:
%pip install -q nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
from itertools import islice
from nltk import CFG, ChartParser, Tree
from nltk.grammar import Nonterminal, Production

# Gujarati is head-final (SOV): arguments, postpositional phrases and adverbs come before the verb group,
# postpositions follow their noun phrase, adjectives/numerals precede the noun, auxiliaries follow the main verb.
# The grammar is written over the Step 3 POS tags (preterminals); the lexical rules (TAG -> 'word')
# are added from whichever tagged sentences are being parsed.
STRUCTURE_RULES = """
S    -> CL | CL PUNCT
CL   -> NP VP | VP | ADVP CL | PART CL | CL CCONJ CL | CL PUNCT CCONJ CL
VP   -> VG | NP VP | PP VP | ADVP VP | ADJP VP | VG COMP
COMP -> CCONJ CL
VG   -> VERB | VERB VG
NP   -> NOM | PRON | PRON NOM | NUMP NOM | PRON NUMP NOM
NOM  -> NOUN | PROPN | NOUN NOM | PROPN NOM | ADJP NOM
ADJP -> ADJ | ADV ADJ
NUMP -> NUM | NUM NUMP
PP   -> NP ADP
ADVP -> ADV | ADV PART
"""
# S = sentence, CL = clause, VP = verb phrase, VG = verb group (main verb + auxiliaries),
# COMP = complement clause introduced by કે, NP = noun phrase, NOM = noun with modifiers / compound noun,
# ADJP = adjective phrase, NUMP = numeral phrase, PP = postpositional phrase, ADVP = adverb phrase
STRUCTURE = CFG.fromstring(STRUCTURE_RULES).productions()
MAX_PARSES = 1000   # cap when enumerating parses of highly ambiguous sentences


def make_parser(tagged_sents):
    lexicon = {Production(Nonterminal(t), [w]) for s in tagged_sents for w, t in s}
    return ChartParser(CFG(Nonterminal('S'), STRUCTURE + sorted(lexicon, key=str)))


def parse(parser, tagged_sent):
    return list(islice(parser.parse([w for w, _ in tagged_sent]), MAX_PARSES))


print(f'{len(STRUCTURE)} structural rules:')
for p in STRUCTURE:
    print('   ', p)

34 structural rules:
    S -> CL
    S -> CL PUNCT
    CL -> NP VP
    CL -> VP
    CL -> ADVP CL
    CL -> PART CL
    CL -> CL CCONJ CL
    CL -> CL PUNCT CCONJ CL
    VP -> VG
    VP -> NP VP
    VP -> PP VP
    VP -> ADVP VP
    VP -> ADJP VP
    VP -> VG COMP
    COMP -> CCONJ CL
    VG -> VERB
    VG -> VERB VG
    NP -> NOM
    NP -> PRON
    NP -> PRON NOM
    NP -> NUMP NOM
    NP -> PRON NUMP NOM
    NOM -> NOUN
    NOM -> PROPN
    NOM -> NOUN NOM
    NOM -> PROPN NOM
    NOM -> ADJP NOM
    ADJP -> ADJ
    ADJP -> ADV ADJ
    NUMP -> NUM
    NUMP -> NUM NUMP
    PP -> NP ADP
    ADVP -> ADV
    ADVP -> ADV PART


In [39]:
# Apply the grammar to the gold-tagged sentences from Step 3
gold_parser = make_parser(gold_sents)
print(f'Grammar: {len(STRUCTURE)} structural + {len(gold_parser.grammar().productions()) - len(STRUCTURE)} lexical rules')

gold_parses = [parse(gold_parser, s) for s in gold_sents]
gold_summary = pd.DataFrame({
    'sentence': [' '.join(w for w, _ in s) for s in gold_sents],
    'tags': [' '.join(t for _, t in s) for s in gold_sents],
    'n_parses': [len(p) for p in gold_parses],
})
print(f"Parsed {(gold_summary['n_parses'] > 0).sum()} / {len(gold_summary)} gold sentences "
      f'(parse counts capped at {MAX_PARSES})')
gold_summary

Grammar: 34 structural + 125 lexical rules
Parsed 20 / 20 gold sentences (parse counts capped at 1000)


,sentence,tags,n_parses
0,રવિવારે ગામમાં આરોગ્ય કેમ્પ યોજાશે .,NOUN NOUN NOUN NOUN VERB PUNCT,16
1,મુખ્યમંત્રીએ આજે નવી યોજનાની જાહેરાત કરી .,NOUN ADV ADJ NOUN NOUN VERB PUNCT,20
2,પોલીસે આરોપીની ધરપકડ કરી હતી .,NOUN NOUN NOUN VERB VERB PUNCT,8
3,અમદાવાદમાં ભારે વરસાદ પડ્યો અને રસ્તાઓ પર પાણી ભરાયા .,PROPN ADJ NOUN VERB CCONJ NOUN ADP NOUN VERB PUNCT,12
4,તેઓ દરરોજ આ મંદિરની મુલાકાત લે છે .,PRON ADV PRON NOUN NOUN VERB VERB PUNCT,8
5,ભારતીય ટીમે ફાઇનલ મેચમાં શાનદાર જીત મેળવી .,ADJ NOUN ADJ NOUN ADJ NOUN VERB PUNCT,27
6,વિદ્યાર્થીઓ માટે શાળામાં મફત પુસ્તકો આપવામાં આવશે .,NOUN ADP NOUN ADJ NOUN VERB VERB PUNCT,3
7,હું કાલે મારા મિત્ર સાથે સુરત જઈશ .,PRON ADV PRON NOUN ADP PROPN VERB PUNCT,4
8,આ ઘટનામાં બે લોકોના મોત થયા છે .,PRON NOUN NUM NOUN NOUN VERB VERB PUNCT,8
9,સરકારે ખેડૂતો માટે ૫૦૦ કરોડ રૂપિયાની સહાય જાહેર કરી .,NOUN NOUN ADP NUM NUM NOUN NOUN ADJ VERB PUNCT,6


In [40]:
# Parse trees: a simple SOV sentence, two coordinated clauses, and a complement (કે) clause
for i in (1, 3, 14):
    print(f"[{i + 1}] {gold_summary.loc[i, 'sentence']}   ({gold_summary.loc[i, 'n_parses']} parses, first shown)")
    if gold_parses[i]:
        gold_parses[i][0].pretty_print()
        print(gold_parses[i][0], '\n')

[2] મુખ્યમંત્રીએ આજે નવી યોજનાની જાહેરાત કરી .   (20 parses, first shown)
                                             S              
                                  ___________|___________    
                                 CL                      |  
                                 |                       |   
                                 VP                      |  
                        _________|_________________      |   
                       NP                          |     |  
                       |                           |     |   
                      NOM                          |     |  
      _________________|___                        |     |   
     |                    NOM                      |     |  
     |             ________|___________            |     |   
     |            |                   NOM          VP    |  
     |            |               _____|_____      |     |   
     |           ADJP            |          NOM    VG    |  
    

In [41]:
# Structural ambiguity: the same tag sequence can be bracketed in several ways,
# e.g. consecutive nouns as one compound NP or as separate arguments of the verb.
i = 0
print(f"{gold_summary.loc[i, 'sentence']}  ->  {gold_summary.loc[i, 'n_parses']} parses. First three:\n")
for t in gold_parses[i][:3]:
    print(t, '\n')

રવિવારે ગામમાં આરોગ્ય કેમ્પ યોજાશે .  ->  16 parses. First three:

(S
  (CL
    (VP
      (NP
        (NOM
          (NOUN રવિવારે)
          (NOM (NOUN ગામમાં) (NOM (NOUN આરોગ્ય) (NOM (NOUN કેમ્પ))))))
      (VP (VG (VERB યોજાશે)))))
  (PUNCT .)) 

(S
  (CL
    (VP
      (NP
        (NOM (NOUN રવિવારે) (NOM (NOUN ગામમાં) (NOM (NOUN આરોગ્ય)))))
      (VP (NP (NOM (NOUN કેમ્પ))) (VP (VG (VERB યોજાશે))))))
  (PUNCT .)) 

(S
  (CL
    (VP
      (NP (NOM (NOUN રવિવારે) (NOM (NOUN ગામમાં))))
      (VP
        (NP (NOM (NOUN આરોગ્ય) (NOM (NOUN કેમ્પ))))
        (VP (VG (VERB યોજાશે))))))
  (PUNCT .)) 



In [42]:
# The grammar accepts Gujarati word order and rejects sequences that violate it
s3, s4 = gold_sents[2], gold_sents[3][5:]
tests = [
    ('SOV order', s3),
    ('verb group moved to the front', s3[3:5] + s3[:3] + s3[5:]),
    ('postposition after its noun', s4),
    ('postposition before its noun', [s4[1], s4[0]] + s4[2:]),
]
for label, s in tests:
    n = len(parse(make_parser([s]), s))
    print(f"{'ACCEPTED' if n else 'REJECTED':8s} | {label:30s} | {' '.join(w for w, _ in s)}")

ACCEPTED | SOV order                      | પોલીસે આરોપીની ધરપકડ કરી હતી .
REJECTED | verb group moved to the front  | કરી હતી પોલીસે આરોપીની ધરપકડ .
ACCEPTED | postposition after its noun    | રસ્તાઓ પર પાણી ભરાયા .
REJECTED | postposition before its noun   | પર રસ્તાઓ પાણી ભરાયા .


In [43]:
# Sentence structure analysis (first parse of each gold sentence): the clause is flattened into its
# ordered constituents, i.e. the arguments/adjuncts followed by the verb group.
def constituents(tree):
    out = []
    for child in tree:
        if isinstance(child, Tree) and child.label() in ('S', 'CL', 'VP'):
            out.extend(constituents(child))
        else:
            out.append(child)
    return out


def count_label(tree, label):
    # maximal subtrees with this label (a VG nested inside a VG counts once)
    return sum(1 for pos in tree.treepositions()
               if isinstance(tree[pos], Tree) and tree[pos].label() == label
               and (len(pos) == 0 or tree[pos[:-1]].label() != label))


rows, rule_use = [], Counter()
for s, parses in zip(gold_sents, gold_parses):
    if not parses:
        continue
    t = parses[0]
    parts = [c for c in constituents(t) if c.label() != 'PUNCT']
    rule_use.update(str(p) for p in t.productions() if p.is_nonlexical())
    rows.append({
        'pattern': ' '.join(c.label() for c in parts),
        'constituents': ' | '.join(f"{c.label()}[{' '.join(c.leaves())}]" for c in parts),
        'verb_final': parts[-1].label() == 'VG',
        'verb_groups': count_label(t, 'VG'),
        'NPs': count_label(t, 'NP'),
        'PPs': count_label(t, 'PP'),
        'tree_height': t.height(),
    })
structure = pd.DataFrame(rows)
print(f"Verb-final sentences: {structure['verb_final'].sum()} / {len(structure)}")
display(structure)
print('Most frequent constituent patterns:')
structure['pattern'].value_counts().head(10)

Verb-final sentences: 20 / 20


,pattern,constituents,verb_final,verb_groups,NPs,PPs,tree_height
0,NP VG,NP[રવિવારે ગામમાં આરોગ્ય કેમ્પ] | VG[યોજાશે],True,1,1,0,10
1,NP VG,NP[મુખ્યમંત્રીએ આજે નવી યોજનાની જાહેરાત] | VG[કરી],True,1,1,0,10
2,NP VG,NP[પોલીસે આરોપીની ધરપકડ] | VG[કરી હતી],True,1,1,0,9
3,NP VG CCONJ PP NP VG,NP[અમદાવાદમાં ભારે વરસાદ] | VG[પડ્યો] | CCONJ[અને] | PP[રસ્તાઓ પર] | NP[પાણી] | VG[ભરાયા],True,2,3,1,10
4,NP ADVP NP VG,NP[તેઓ] | ADVP[દરરોજ] | NP[આ મંદિરની મુલાકાત] | VG[લે છે],True,1,2,0,10
5,ADJP NP VG,ADJP[ભારતીય] | NP[ટીમે ફાઇનલ મેચમાં શાનદાર જીત] | VG[મેળવી],True,1,1,0,12
6,PP NP VG,PP[વિદ્યાર્થીઓ માટે] | NP[શાળામાં મફત પુસ્તકો] | VG[આપવામાં આવશે],True,1,2,1,10
7,NP ADVP PP NP VG,NP[હું] | ADVP[કાલે] | PP[મારા મિત્ર સાથે] | NP[સુરત] | VG[જઈશ],True,1,3,1,10
8,NP NP VG,NP[આ ઘટનામાં] | NP[બે લોકોના મોત] | VG[થયા છે],True,1,2,0,9
9,PP NP ADJP VG,PP[સરકારે ખેડૂતો માટે] | NP[૫૦૦ કરોડ રૂપિયાની સહાય] | ADJP[જાહેર] | VG[કરી],True,1,2,1,9


Most frequent constituent patterns:


pattern
NP VG                          5
NP ADVP NP VG                  2
PP NP VG                       2
NP NP VG                       2
NP VG CCONJ PP NP VG           1
ADJP NP VG                     1
NP ADVP PP NP VG               1
PP NP ADJP VG                  1
NP ADJP VG                     1
NP VG CCONJ NP ADVP ADJP VG    1
Name: count, dtype: int64

In [44]:
# How often each structural rule is used across the gold parses
pd.DataFrame(rule_use.most_common(), columns=['rule', 'times used'])

,rule,times used
0,VP -> NP VP,29
1,NOM -> NOUN,28
2,NOM -> NOUN NOM,27
3,CL -> VP,23
4,VP -> VG,23
5,VG -> VERB,23
6,NP -> NOM,21
7,S -> CL PUNCT,20
8,ADJP -> ADJ,15
9,NOM -> ADJP NOM,11


In [45]:
# Apply the grammar to real corpus sentences (3-10 tokens) tagged by the Step 3 HMM
corpus_tagged = [s for s in hmm_test_pred if 3 <= len(s) <= 10][:300]
res = []
for s in tqdm(corpus_tagged, desc='Parsing'):
    tags = [t for _, t in s]
    content = [t for t in tags if t != 'PUNCT']
    res.append({
        'sentence': ' '.join(w for w, _ in s),
        'tags': ' '.join(tags),
        'n_parses': len(parse(make_parser([s]), s)),
        'ends_in_verb': bool(content) and content[-1] == 'VERB',
        'inner_punct': 'PUNCT' in tags[:-1],
    })
corpus_res = pd.DataFrame(res)
parsed = corpus_res['n_parses'] > 0
print(f'Corpus sentences parsed: {parsed.sum()} / {len(corpus_res)} ({parsed.mean():.1%})')
print(f"Average parses per parsed sentence: {corpus_res.loc[parsed, 'n_parses'].mean():.1f} (capped at {MAX_PARSES})")
print('\nCoverage by sentence property:')
display(pd.crosstab([corpus_res['ends_in_verb'], corpus_res['inner_punct']], parsed.rename('parsed'), margins=True))
print('Examples the grammar could not parse:')
corpus_res.loc[~parsed, ['sentence', 'tags']].head(10)

Parsing: 100%|██████████| 300/300 [00:00<00:00, 670.32it/s]

Corpus sentences parsed: 88 / 300 (29.3%)
Average parses per parsed sentence: 43.8 (capped at 1000)

Coverage by sentence property:


parsed                    False  True  All
ends_in_verb inner_punct                  
False        False           79     0   79
             True            11     0   11
True         False          114    88  202
             True             8     0    8
All                         212    88  300

Examples the grammar could not parse:


,sentence,tags
0,"અમે લગભગ 5,300 ગ્રાહકોને વીજળી વગર જોયા છે .",PRON NOUN NOUN VERB NOUN NOUN VERB VERB PUNCT
1,"પરંતુ ત્યાં સુધી , ફરી , તમે ઘરે રહો .",NOUN ADV ADP PUNCT ADV PUNCT PRON VERB NOUN PUNCT
2,મેટ્રોથી દૂર રહો .,NOUN NOUN NOUN PUNCT
3,રસ્તાઓથી દૂર રહો .,NOUN NOUN NOUN PUNCT
4,આ ભારે પાણીમાં વાહન ન ચલાવો .,PRON VERB NOUN NOUN NOUN NOUN PUNCT
5,કાર્યક્રમનું જીવંત પ્રસારણ કરવામાં આવ્યું હતું .,NOUN VERB NOUN NOUN VERB NOUN PUNCT
6,એક અધિકારીએ કહ્યું અમે મામલાની તપાસ કરી રહ્યા છીએ .,NOUN NOUN VERB PRON NOUN NOUN VERB VERB ADV PUNCT
7,તેની વિગત આપો .,NOUN VERB NOUN PUNCT
9,તેવું શેરબજારનો ટ્રેન્ડ કહી રહ્યો છે .,VERB NOUN NOUN NOUN VERB VERB PUNCT
11,અકસ્માતનું કારણ શું છે તે હજુ સ્પષ્ટ થયું નથી .,NOUN CCONJ PRON VERB PRON NOUN NOUN VERB ADV PUNCT


### Step 5: Probabilistic Parsing

In [47]:
from nltk import PCFG, ViterbiParser
from nltk.grammar import ProbabilisticProduction

MAX_TRAIN_PARSES = 200
N_TRAIN_CANDIDATES = 4000

rule_index = {p: i for i, p in enumerate(STRUCTURE)}
R = len(STRUCTURE)


def rule_counts(tree):
    v = np.zeros(R)
    for p in tree.productions():
        if p.is_nonlexical():
            v[rule_index[p]] += 1
    return v


candidates = [s for s in pos_train if 3 <= len(s) <= 12][:N_TRAIN_CANDIDATES]
forests, no_parse, too_ambiguous = [], 0, 0
for s in tqdm(candidates, desc='Parsing training sentences'):
    trees = list(islice(make_parser([s]).parse([w for w, _ in s]), MAX_TRAIN_PARSES + 1))
    if not trees:
        no_parse += 1
    elif len(trees) > MAX_TRAIN_PARSES:
        too_ambiguous += 1
    else:
        forests.append(np.array([rule_counts(t) for t in trees]))

print(f'Candidates: {len(candidates)} | used for training: {len(forests)} | '
      f'no parse: {no_parse} | > {MAX_TRAIN_PARSES} parses: {too_ambiguous}')
print(f'Total parse trees in training forests: {sum(len(f) for f in forests)}')

Parsing training sentences: 100%|██████████| 4000/4000 [00:08<00:00, 481.13it/s]

Candidates: 4000 | used for training: 748 | no parse: 3201 | > 200 parses: 51
Total parse trees in training forests: 27748


In [48]:
# EM training of the rule probabilities
lhs_of = np.array([str(p.lhs()) for p in STRUCTURE])
lhs_groups = [np.where(lhs_of == a)[0] for a in sorted(set(lhs_of))]
C = np.vstack(forests)                                      # (all parses) x (rules) count matrix
starts = np.cumsum([0] + [len(f) for f in forests[:-1]])    # first row of each sentence's forest
sizes = np.array([len(f) for f in forests])


def normalise(counts, smoothing=0.1):
    theta = np.empty(R)
    for idx in lhs_groups:
        c = counts[idx] + smoothing
        theta[idx] = c / c.sum()
    return theta


def em_step(theta):
    logp = C @ np.log(theta)                                # log P(tree) for every parse
    seg_max = np.maximum.reduceat(logp, starts)
    w = np.exp(logp - np.repeat(seg_max, sizes))
    z = np.add.reduceat(w, starts)
    loglik = np.sum(np.log(z) + seg_max)                    # sum over sentences of log P(sentence)
    posterior = w / np.repeat(z, sizes)                     # P(tree | sentence)
    return normalise(posterior @ C), loglik


uniform = normalise(np.zeros(R), smoothing=1.0)
theta, history = uniform, []
for _ in range(20):
    theta, ll = em_step(theta)
    history.append(ll)

print('Corpus log-likelihood by EM iteration:')
print(pd.Series(history, index=range(1, len(history) + 1)).round(1).to_string())

rule_probs = pd.DataFrame({'rule': [str(p) for p in STRUCTURE], 'uniform': uniform, 'learned': theta}).round(3)
rule_probs

Corpus log-likelihood by EM iteration:
1    -12455.4
2     -5946.0
3     -5850.4
4     -5811.4
5     -5790.6
6     -5777.4
7     -5768.3
8     -5761.6
9     -5756.4
10    -5752.4
11    -5749.1
12    -5746.5
13    -5744.3
14    -5742.4
15    -5740.8
16    -5739.5
17    -5738.3
18    -5737.3
19    -5736.5
20    -5735.7


,rule,uniform,learned
0,S -> CL,0.500,0.003
1,S -> CL PUNCT,0.500,0.997
2,CL -> NP VP,0.167,0.799
3,CL -> VP,0.167,0.173
4,CL -> ADVP CL,0.167,0.022
5,CL -> PART CL,0.167,0.000
6,CL -> CL CCONJ CL,0.167,0.004
7,CL -> CL PUNCT CCONJ CL,0.167,0.001
8,VP -> VG,0.167,0.293
9,VP -> NP VP,0.167,0.593


In [49]:
# PCFG parser for a set of tagged sentences: learned structural probabilities + lexical rules TAG -> word.
# Lexical probabilities are uniform per tag; since each word has one tag in a tagged sentence, they are the
# same for every parse of that sentence and do not affect which tree is most likely.
# ViterbiParser is a CKY-style dynamic-programming parser that returns the single most probable tree.
def make_pcfg_parser(tagged_sents):
    prods = [ProbabilisticProduction(p.lhs(), p.rhs(), prob=theta[i]) for i, p in enumerate(STRUCTURE)]
    words_by_tag = defaultdict(set)
    for s in tagged_sents:
        for w, t in s:
            words_by_tag[t].add(w)
    for t, words in words_by_tag.items():
        prods += [ProbabilisticProduction(Nonterminal(t), [w], prob=1 / len(words)) for w in sorted(words)]
    return ViterbiParser(PCFG(Nonterminal('S'), prods))


def parse_posteriors(parses):
    """P(tree | sentence) for every enumerated parse, highest first."""
    logp = np.array([rule_counts(t) @ np.log(theta) for t in parses])
    post = np.exp(logp - logp.max())
    post /= post.sum()
    order = np.argsort(-post)
    return [parses[i] for i in order], post[order]


def flat(tree):
    return tree.pformat(margin=10 ** 6)

In [50]:
# Most likely parse of each gold sentence
pcfg_gold = make_pcfg_parser(gold_sents)
best_trees, rows = [], []
for s, parses in zip(gold_sents, gold_parses):
    best = next(pcfg_gold.parse([w for w, _ in s]), None)
    best_trees.append(best)
    if best is None:
        rows.append({'sentence': ' '.join(w for w, _ in s), 'n_parses': 0})
        continue
    ranked, post = parse_posteriors(parses)
    rows.append({
        'sentence': ' '.join(w for w, _ in s),
        'n_parses': len(parses),
        'log10 P(best tree)': round(best.logprob() * math.log10(2), 2),   # nltk logprob is base 2
        'P(best | sentence)': round(post[0], 3),
        'best / 2nd best': round(post[0] / post[1], 1) if len(post) > 1 else None,
        'Viterbi = top enumerated': flat(best) == flat(ranked[0]),
        'same as Step 4 first parse': flat(best) == flat(parses[0]),
    })

pcfg_summary = pd.DataFrame(rows)
print(f"Viterbi tree matches the highest-probability enumerated tree: "
      f"{pcfg_summary['Viterbi = top enumerated'].sum()} / {len(pcfg_summary)}")
print(f"PCFG choice differs from the arbitrary first CFG parse of Step 4: "
      f"{(~pcfg_summary['same as Step 4 first parse'].astype(bool)).sum()} / {len(pcfg_summary)}")
pcfg_summary

Viterbi tree matches the highest-probability enumerated tree: 20 / 20
PCFG choice differs from the arbitrary first CFG parse of Step 4: 20 / 20


,sentence,n_parses,log10 P(best tree),P(best | sentence),best / 2nd best,Viterbi = top enumerated,same as Step 4 first parse
0,રવિવારે ગામમાં આરોગ્ય કેમ્પ યોજાશે .,16,-11.19,0.119,1.1,True,False
1,મુખ્યમંત્રીએ આજે નવી યોજનાની જાહેરાત કરી .,20,-14.31,0.188,1.1,True,False
2,પોલીસે આરોપીની ધરપકડ કરી હતી .,8,-10.78,0.233,1.1,True,False
3,અમદાવાદમાં ભારે વરસાદ પડ્યો અને રસ્તાઓ પર પાણી ભરાયા .,12,-21.85,0.478,2.2,True,False
4,તેઓ દરરોજ આ મંદિરની મુલાકાત લે છે .,8,-15.75,0.320,1.1,True,False
5,ભારતીય ટીમે ફાઇનલ મેચમાં શાનદાર જીત મેળવી .,27,-18.18,0.264,2.2,True,False
6,વિદ્યાર્થીઓ માટે શાળામાં મફત પુસ્તકો આપવામાં આવશે .,3,-16.12,0.648,3.6,True,False
7,હું કાલે મારા મિત્ર સાથે સુરત જઈશ .,4,-17.20,0.625,2.4,True,False
8,આ ઘટનામાં બે લોકોના મોત થયા છે .,8,-14.63,0.320,1.1,True,False
9,સરકારે ખેડૂતો માટે ૫૦૦ કરોડ રૂપિયાની સહાય જાહેર કરી .,6,-21.15,0.410,1.1,True,False


In [51]:
# Most likely parse trees for the same three sentences shown in Step 4
for i in (1, 3, 14):
    best = best_trees[i]
    if best is None:
        continue
    print(f"[{i + 1}] {pcfg_summary.loc[i, 'sentence']}")
    print(f"     P(tree) = {best.prob():.3e}   P(tree | sentence) = {pcfg_summary.loc[i, 'P(best | sentence)']}")
    best.pretty_print()
    print(flat(best), '\n')

[2] મુખ્યમંત્રીએ આજે નવી યોજનાની જાહેરાત કરી .
     P(tree) = 4.922e-15   P(tree | sentence) = 0.188
                             S                          
                        _____|_______________________    
                       CL                            |  
      _________________|_____                        |   
     |                       VP                      |  
     |             __________|_______                |   
     |            |                  VP              |  
     |            |           _______|_____          |   
     |            |          |             VP        |  
     |            |          |        _____|___      |   
     NP           |          NP      NP        VP    |  
     |            |          |       |         |     |   
    NOM          ADJP       NOM     NOM        VG    |  
     |         ___|____      |       |         |     |   
    NOUN     ADV      ADJ   NOUN    NOUN      VERB PUNCT
     |        |        |     |       

In [52]:
# Ambiguity resolution: all parses of an ambiguous sentence ranked by probability
def show_ranked(i, k=5):
    ranked, post = parse_posteriors(gold_parses[i])
    print(f"[{i + 1}] {pcfg_summary.loc[i, 'sentence']}  ->  {len(ranked)} parses, top {min(k, len(ranked))}:\n")
    for rank, (t, p) in enumerate(zip(ranked[:k], post[:k]), 1):
        print(f'  #{rank}  P(tree | sentence) = {p:.3f}')
        print(f'      {flat(t)}\n')


ambiguous = pcfg_summary['n_parses'].nlargest(2).index
for i in ambiguous:
    show_ranked(i)

[13] પ્રધાનમંત્રી નરેન્દ્ર મોદીએ ગાંધીનગરમાં નવા પુલનું ઉદ્ઘાટન કર્યું .  ->  96 parses, top 5:

  #1  P(tree | sentence) = 0.052
      (S (CL (NP (NOM (NOUN પ્રધાનમંત્રી))) (VP (NP (NOM (PROPN નરેન્દ્ર) (NOM (PROPN મોદીએ) (NOM (PROPN ગાંધીનગરમાં))))) (VP (ADJP (ADJ નવા)) (VP (NP (NOM (NOUN પુલનું))) (VP (NP (NOM (NOUN ઉદ્ઘાટન))) (VP (VG (VERB કર્યું)))))))) (PUNCT .))

  #2  P(tree | sentence) = 0.050
      (S (CL (NP (NOM (NOUN પ્રધાનમંત્રી) (NOM (PROPN નરેન્દ્ર) (NOM (PROPN મોદીએ) (NOM (PROPN ગાંધીનગરમાં)))))) (VP (ADJP (ADJ નવા)) (VP (NP (NOM (NOUN પુલનું))) (VP (NP (NOM (NOUN ઉદ્ઘાટન))) (VP (VG (VERB કર્યું))))))) (PUNCT .))

  #3  P(tree | sentence) = 0.050
      (S (CL (NP (NOM (NOUN પ્રધાનમંત્રી))) (VP (NP (NOM (PROPN નરેન્દ્ર) (NOM (PROPN મોદીએ) (NOM (PROPN ગાંધીનગરમાં))))) (VP (ADJP (ADJ નવા)) (VP (NP (NOM (NOUN પુલનું) (NOM (NOUN ઉદ્ઘાટન)))) (VP (VG (VERB કર્યું))))))) (PUNCT .))

  #4  P(tree | sentence) = 0.047
      (S (CL (NP (NOM (NOUN પ્રધાનમંત્રી) (NOM (PROPN નરેન્દ્ર

In [53]:
# Apply the PCFG to the HMM-tagged corpus sentences that the CFG parsed in Step 4
corpus_rows = []
for s, n in zip(corpus_tagged, corpus_res['n_parses']):
    if not 0 < n < MAX_PARSES:
        continue
    parses = parse(make_parser([s]), s)
    ranked, post = parse_posteriors(parses)
    corpus_rows.append({
        'sentence': ' '.join(w for w, _ in s),
        'n_parses': len(parses),
        'P(best | sentence)': round(post[0], 3),
        'most likely tree': flat(ranked[0]),
    })
corpus_pcfg = pd.DataFrame(corpus_rows)

print(f'Sentences: {len(corpus_pcfg)}')
print(f"Unambiguous (1 parse): {(corpus_pcfg['n_parses'] == 1).sum()}")
print(f"Mean P(best | sentence) for ambiguous sentences: "
      f"{corpus_pcfg.loc[corpus_pcfg['n_parses'] > 1, 'P(best | sentence)'].mean():.3f}")
print('\nMost ambiguous sentences and the tree the PCFG chooses:')
corpus_pcfg.sort_values('n_parses', ascending=False).head(10)

Sentences: 88
Unambiguous (1 parse): 0
Mean P(best | sentence) for ambiguous sentences: 0.195

Most ambiguous sentences and the tree the PCFG chooses:


,sentence,n_parses,P(best | sentence),most likely tree
82,બોર્ડની વેબસાઈટ પરથી તમામ વિદ્યાર્થીઓ પોતાનું પરિણામ જોઇ શકશે .,256,0.008,(S (CL (NP (NOM (NOUN બોર્ડની))) (VP (NP (NOM (NOUN વેબસાઈટ))) (VP (NP (NOM (NOUN પરથી))) (VP (NP (NOM (NOUN તમામ))) (VP (NP (NOM (NOUN વિદ્યાર્થીઓ))) (VP (NP (NOM (NOUN પોતાનું))) (VP (NP (NOM (N...
37,આ દરમિયાન તેની સરેરાશ માત્ર 20ની આસપાસ રહી છે .,256,0.011,(S (CL (NP (PRON આ) (NOM (NOUN દરમિયાન))) (VP (NP (NOM (NOUN તેની))) (VP (NP (NOM (NOUN સરેરાશ))) (VP (NP (NOM (NOUN માત્ર))) (VP (NP (NOM (NOUN 20ની))) (VP (NP (NOM (NOUN આસપાસ))) (VP (NP (NOM (N...
81,તાલિબાન સામેનો સૌથી મોટો ખતરો ઇસ્લામિક સ્ટેટ જૂથ છે .,192,0.020,(S (CL (NP (NOM (NOUN તાલિબાન))) (VP (NP (NOM (NOUN સામેનો))) (VP (NP (NOM (NOUN સૌથી))) (VP (ADJP (ADJ મોટો)) (VP (NP (NOM (NOUN ખતરો))) (VP (NP (NOM (NOUN ઇસ્લામિક))) (VP (NP (NOM (NOUN સ્ટેટ)))...
31,બધા પહેલા ચરાગવાન વિસ્તારમાં રામુજી વોટર પાર્ક પહોંચ્યા .,128,0.016,(S (CL (NP (NOM (NOUN બધા))) (VP (NP (NOM (NOUN પહેલા))) (VP (NP (NOM (NOUN ચરાગવાન))) (VP (NP (NOM (NOUN વિસ્તારમાં))) (VP (NP (NOM (NOUN રામુજી))) (VP (NP (NOM (NOUN વોટર))) (VP (NP (NOM (NOUN પ...
25,પરંતુ આ દરમિયાન દેશમાં બેરોજગારી ઝડપથી વધી રહી હતી .,128,0.022,(S (CL (NP (NOM (NOUN પરંતુ))) (VP (NP (PRON આ) (NOM (NOUN દરમિયાન))) (VP (NP (NOM (NOUN દેશમાં))) (VP (NP (NOM (NOUN બેરોજગારી))) (VP (NP (NOM (NOUN ઝડપથી))) (VP (NP (NOM (NOUN વધી))) (VP (NP (NO...
34,ઘરવાપસી કરનારા મોટા ભાગના પરિવાર બસના સરાઈપાલીના હતા .,128,0.016,(S (CL (NP (NOM (NOUN ઘરવાપસી))) (VP (NP (NOM (NOUN કરનારા))) (VP (NP (NOM (NOUN મોટા))) (VP (NP (NOM (NOUN ભાગના))) (VP (NP (NOM (NOUN પરિવાર))) (VP (NP (NOM (NOUN બસના))) (VP (NP (NOM (NOUN સરાઈ...
68,વ્યવસાયના દ્રષ્ટિકોણથી ડીઝલનું ઉત્પાદન પેટ્રોલ કરતાં મોંઘું છે .,128,0.016,(S (CL (NP (NOM (NOUN વ્યવસાયના))) (VP (NP (NOM (NOUN દ્રષ્ટિકોણથી))) (VP (NP (NOM (NOUN ડીઝલનું))) (VP (NP (NOM (NOUN ઉત્પાદન))) (VP (NP (NOM (NOUN પેટ્રોલ))) (VP (NP (NOM (NOUN કરતાં))) (VP (NP ...
47,મેયર મત લઈ ગયા તો કેમ કામ નથી કરતા ?,128,0.051,(S (CL (NP (NOM (NOUN મેયર))) (VP (NP (NOM (NOUN મત))) (VP (NP (NOM (NOUN લઈ))) (VP (VG (VERB ગયા)) (COMP (CCONJ તો) (CL (NP (NOM (NOUN કેમ))) (VP (NP (NOM (NOUN કામ))) (VP (NP (NOM (NOUN નથી))) (...
77,આ દર્દનાક અકસ્માતમાં બે અભિનેતાએ પોતાનો જીવ ગુમાવ્યો હતો .,128,0.022,(S (CL (NP (PRON આ) (NOM (NOUN દર્દનાક))) (VP (NP (NOM (NOUN અકસ્માતમાં))) (VP (NP (NOM (NOUN બે))) (VP (NP (NOM (NOUN અભિનેતાએ))) (VP (NP (NOM (NOUN પોતાનો))) (VP (NP (NOM (NOUN જીવ))) (VP (VG (V...
74,સોમનાથના માર્ગો શિવભક્તોનાં જય સોમનાથના નાદથી ગુંજી ઉઠશે .,128,0.016,(S (CL (NP (NOM (NOUN સોમનાથના))) (VP (NP (NOM (NOUN માર્ગો))) (VP (NP (NOM (NOUN શિવભક્તોનાં))) (VP (NP (NOM (NOUN જય))) (VP (NP (NOM (NOUN સોમનાથના))) (VP (NP (NOM (NOUN નાદથી))) (VP (NP (NOM (N...


### Step 6: Semantic Analysis using WordNet

In [54]:
%pip install -q pyiwn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
# Gujarati WordNet = IndoWordNet, accessed through pyiwn. Importing pyiwn downloads the IndoWordNet
# data (~31 MB) into ~/iwn_data the first time. Semantic relations (hypernymy, hyponymy, ...) are shared
# across the IndoWordNet languages through common synset ids.
from pyiwn import IndoWordNet, Language, PosTag, SynsetRelations

iwn = IndoWordNet(lang=Language.GUJARATI)
WN_WORDS = set(iwn.all_words())

pd.Series({
    'Synsets': len(iwn.all_synsets()),
    'Words (lemmas)': len(WN_WORDS),
    **{f'{p.value.capitalize()} synsets': len(iwn.all_synsets(p)) for p in PosTag},
}).to_frame('Gujarati IndoWordNet')

2026-10-09:22:39:32,401 INFO     [helpers.py:20] Downloading IndoWordNet data of size ~31 MB...


[██████████████████████████████████████████████████]

2026-10-09:22:39:45,631 INFO     [helpers.py:43] Extracting C:\Users\Samyak\iwn_data.tar.gz into C:\Users\Samyak...


2026-10-09:22:39:50,142 INFO     [helpers.py:48] Removing temporary zip file from C:\Users\Samyak\iwn_data.tar.gz
2026-10-09:22:39:50,156 INFO     [helpers.py:51] IndoWordNet data successfully downloaded at C:\Users\Samyak\iwn_data
2026-10-09:22:39:50,174 INFO     [iwn.py:43] Loading gujarati language synsets...


UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 68: character maps to <undefined>

In [ ]:
# WordNet stores base forms (ઘર, શહેર, મોટો) while news text is inflected (ઘરમાં, શહેરના, મોટા).
# wn_lemma() strips case/plural suffixes and swaps the gender/number endings (-ો/-ું/-ી/-ા)
# until it reaches a form that exists in WordNet.
WN_SUFFIXES = sorted(['માંથી', 'માં', 'નાં', 'ના', 'ની', 'નું', 'નો', 'ને', 'થી', 'એ', 'ઓ', 'ે', 'ો', 'પર'],
                     key=len, reverse=True)
GENDER_ENDINGS = ['ું', 'ો', 'ી', 'ા']


@lru_cache(maxsize=None)
def wn_lemma(word):
    frontier = [word]
    for _ in range(3):
        nxt = []
        for w in frontier:
            if w in WN_WORDS:
                return w
            for e in GENDER_ENDINGS:
                if w.endswith(e):
                    nxt += [w[:-len(e)] + e2 for e2 in GENDER_ENDINGS if e2 != e]
                    break
            nxt += [w[:-len(s)] for s in WN_SUFFIXES if w.endswith(s) and len(w) > len(s) + 1]
        frontier = nxt
    return None


def head_words(synsets, k=None):
    return [s.head_word() for s in synsets][:k]


@lru_cache(maxsize=None)
def hypernyms(synset):
    return tuple(iwn.synset_relation(synset, SynsetRelations.HYPERNYMY))


@lru_cache(maxsize=None)
def hyponyms(synset):
    return tuple(iwn.synset_relation(synset, SynsetRelations.HYPONYMY))


def describe(word, max_senses=3):
    lemma = wn_lemma(word)
    if lemma is None:
        print(f'{word}: not found in IndoWordNet\n')
        return
    senses = iwn.synsets(lemma)
    print(f'{word}  (lemma: {lemma}, {len(senses)} senses)')
    for s in senses[:max_senses]:
        print(f'  [{s.pos()}] {s.gloss()}')
        print(f'      synonyms : {", ".join(n for n in s.lemma_names() if n != lemma) or "-"}')
        if s.examples():
            print(f'      example  : {s.examples()[0]}')
        print(f'      hypernyms: {", ".join(head_words(hypernyms(s))) or "-"}')
        print(f'      hyponyms : {", ".join(head_words(hyponyms(s), 8)) or "-"}')
    print()


for w in ['ઘર', 'પાણી', 'શિક્ષક', 'સારો', 'ખાવું']:
    describe(w)

In [ ]:
# Link the corpus vocabulary (Step 1, stopwords removed) to WordNet
TOP_N = 20000
top_words = final_vocab.most_common(TOP_N)
lemma_freq, direct, via_lemma = Counter(), 0, 0
for w, c in top_words:
    lemma = wn_lemma(w)
    if lemma is None:
        continue
    lemma_freq[lemma] += c
    if lemma == w:
        direct += c
    else:
        via_lemma += c

total = sum(c for _, c in top_words)
n_senses = pd.Series({l: len(iwn.synsets(l)) for l in lemma_freq})
print(f'Token coverage of the top {TOP_N} words: {direct / total:.1%} found directly, '
      f'{via_lemma / total:.1%} after suffix stripping, {1 - (direct + via_lemma) / total:.1%} not in WordNet')
print(f'Distinct WordNet lemmas used in the corpus: {len(lemma_freq)}')
print(f'Average senses per lemma (polysemy): {n_senses.mean():.2f}  |  monosemous lemmas: {(n_senses == 1).mean():.1%}')
print('Most polysemous corpus lemmas:', ', '.join(f'{l} ({n})' for l, n in n_senses.nlargest(10).items()))

In [ ]:
# SYNONYMY: words in the same synset are synonyms. Synonyms per sense of the most frequent corpus lemmas:
rows = []
for lemma, c in lemma_freq.most_common(12):
    senses = iwn.synsets(lemma)
    for s in senses[:2]:
        rows.append({'lemma': lemma, 'corpus freq': c, 'senses': len(senses), 'POS': s.pos(),
                     'synonyms': ', '.join(n for n in s.lemma_names() if n != lemma)[:100],
                     'gloss': s.gloss()[:70]})
pd.DataFrame(rows)

In [ ]:
# Synonyms that actually co-occur in the news corpus: synsets with two or more of their lemmas used
seen, rows = set(), []
for lemma in list(lemma_freq):
    for s in iwn.synsets(lemma):
        if s.synset_id() in seen:
            continue
        seen.add(s.synset_id())
        used = sorted(((n, lemma_freq[n]) for n in s.lemma_names() if lemma_freq[n] > 0), key=lambda x: -x[1])
        if len(used) >= 2:
            rows.append({'synset': s.head_word(), 'POS': s.pos(), 'gloss': s.gloss()[:60],
                         'synonyms used (freq)': ', '.join(f'{n} ({c})' for n, c in used),
                         'total freq': sum(c for _, c in used)})
syn_df = pd.DataFrame(rows).sort_values('total freq', ascending=False).reset_index(drop=True)
print(f'{len(syn_df)} synsets have two or more synonyms used in the corpus')
syn_df.head(15)

In [ ]:
# HYPERNYMY ("is-a"): follow hypernym links upward from the first noun sense to the top of the hierarchy
def hypernym_chain(synset, max_depth=15):
    chain, seen_ids = [synset], {synset.synset_id()}
    while len(chain) < max_depth:
        parents = [p for p in hypernyms(chain[-1]) if p.synset_id() not in seen_ids]
        if not parents:
            break
        chain.append(parents[0])
        seen_ids.add(parents[0].synset_id())
    return chain


def first_noun_sense(word):
    lemma = wn_lemma(word)
    nouns = iwn.synsets(lemma, pos=PosTag.NOUN) if lemma else []
    return nouns[0] if nouns else None


print('Hypernym chains:')
for w in ['કૂતરો', 'ગાય', 'ગુલાબ', 'શિક્ષક', 'બસ', 'નદી']:
    s = first_noun_sense(w)
    print(f'  {w}: ' + ('  →  '.join(x.head_word() for x in hypernym_chain(s)) if s else 'no noun sense in IndoWordNet'))

print('\nHyponyms (more specific concepts):')
for w in ['પ્રાણી', 'ફળ', 'વાહન']:
    s = first_noun_sense(w)
    if s:
        hypo = hyponyms(s)
        print(f'  {w} ({len(hypo)} hyponyms): {", ".join(head_words(hypo, 15))}')

In [ ]:
# Hypernymy-based semantic similarity (path similarity):
#   sim = 1 / (1 + shortest path between two senses through their lowest common hypernym)
@lru_cache(maxsize=None)
def ancestors(synset, max_depth=20):
    dist, by_id, frontier = {synset.synset_id(): 0}, {synset.synset_id(): synset}, [synset]
    for d in range(1, max_depth + 1):
        nxt = []
        for s in frontier:
            for p in hypernyms(s):
                if p.synset_id() not in dist:
                    dist[p.synset_id()] = d
                    by_id[p.synset_id()] = p
                    nxt.append(p)
        frontier = nxt
        if not frontier:
            break
    return dist, by_id


def path_similarity(s1, s2):
    (d1, ids1), (d2, _) = ancestors(s1), ancestors(s2)
    common = set(d1) & set(d2)
    if not common:
        return 0.0, None
    lcs = min(common, key=lambda i: d1[i] + d2[i])
    return 1 / (1 + d1[lcs] + d2[lcs]), ids1[lcs]


def word_similarity(w1, w2, pos=PosTag.NOUN, max_senses=3):
    """Best path similarity over the senses of two words (same POS) and their lowest common hypernym"""
    l1, l2 = wn_lemma(w1), wn_lemma(w2)
    best = (0.0, None)
    if not l1 or not l2:
        return best
    for s1 in iwn.synsets(l1, pos=pos)[:max_senses]:
        for s2 in iwn.synsets(l2, pos=pos)[:max_senses]:
            if s1.pos() == s2.pos():
                best = max(best, path_similarity(s1, s2), key=lambda x: x[0])
    return best


sim_words = ['કૂતરો', 'બિલાડી', 'ગાય', 'ઘોડો', 'શિક્ષક', 'વકીલ', 'નદી', 'પર્વત']
for a, b in [('કૂતરો', 'બિલાડી'), ('કૂતરો', 'શિક્ષક'), ('નદી', 'પર્વત')]:
    score, lcs = word_similarity(a, b)
    print(f'{a} – {b}: similarity {score:.3f}, lowest common hypernym: {lcs.head_word() if lcs else "-"}')

pd.DataFrame([[round(word_similarity(a, b)[0], 3) for b in sim_words] for a in sim_words],
             index=sim_words, columns=sim_words)

In [ ]:
# ANTONYMY. pyiwn does not provide antonymy (Lemma.antonym() is not implemented and the downloaded
# data has no antonym relation), so antonyms are found in two ways:
# 1) Derivational antonyms - Gujarati forms many opposites with negative prefixes:
#    અ-/અન- (સફળ → અસફળ), બિન- (જરૂરી → બિનજરૂરી), ના- (ખુશ → નાખુશ), નિર્- (દોષ → નિર્દોષ),
#    and the સુ-/કુ- (good/bad) pair. A pair is kept only if both words are in WordNet with a shared POS.
NEG_PREFIXES = ['અ', 'અન', 'બિન', 'ના', 'નિર્']


def pos_set(w):
    return {s.pos() for s in iwn.synsets(w)}


def first_gloss(w, pos):
    return next((s.gloss() for s in iwn.synsets(w) if s.pos() == pos), '')


rows = []
for base in WN_WORDS:
    if len(base) < 3:
        continue
    pairs = [(base, pre + base, pre + '-') for pre in NEG_PREFIXES]
    pairs.append(('સુ' + base, 'કુ' + base, 'સુ-/કુ-'))
    for w, ant, rule in pairs:
        if w in WN_WORDS and ant in WN_WORDS:
            shared = pos_set(w) & pos_set(ant)
            if shared:
                pos = sorted(shared)[0]
                rows.append({'word': w, 'antonym': ant, 'rule': rule, 'POS': pos,
                             'freq(word)': lemma_freq[w], 'freq(antonym)': lemma_freq[ant],
                             'gloss(word)': first_gloss(w, pos)[:50], 'gloss(antonym)': first_gloss(ant, pos)[:50]})

deriv_ant = pd.DataFrame(rows)
print(f'Derivational antonym candidates in IndoWordNet: {len(deriv_ant)}')
print(deriv_ant['rule'].value_counts().to_string(), '\n')
print('Pairs where both words occur in the corpus (most frequent first):')
both_used = deriv_ant[(deriv_ant['freq(word)'] > 0) & (deriv_ant['freq(antonym)'] > 0)]
both_used.assign(total=both_used['freq(word)'] + both_used['freq(antonym)']) \
    .sort_values('total', ascending=False).head(15)

In [ ]:
# 2) Lexical (non-derivational) antonyms: a curated list of common opposites, checked against WordNet
#    and the corpus. Antonyms are opposite in meaning but usually close in the hierarchy (same hypernym),
#    so the hypernymy-based similarity and lowest common hypernym are shown as well.
LEXICAL_ANTONYMS = [
    ('સારો', 'ખરાબ'), ('મોટો', 'નાનો'), ('નવો', 'જૂનો'), ('ગરમ', 'ઠંડો'), ('ઊંચો', 'નીચો'),
    ('સાચો', 'ખોટો'), ('ગરીબ', 'અમીર'), ('દિવસ', 'રાત'), ('સવાર', 'સાંજ'), ('જીત', 'હાર'),
    ('સુખ', 'દુઃખ'), ('જન્મ', 'મૃત્યુ'), ('શરૂઆત', 'અંત'), ('પ્રેમ', 'નફરત'), ('મિત્ર', 'દુશ્મન'),
    ('લાભ', 'નુકસાન'), ('ખરીદવું', 'વેચવું'), ('આવવું', 'જવું'),
]

rows = []
for a, b in LEXICAL_ANTONYMS:
    la, lb = wn_lemma(a), wn_lemma(b)
    score, lcs = word_similarity(a, b, pos=None)
    rows.append({'word': a, 'antonym': b, 'lemma(word)': la or '-', 'lemma(antonym)': lb or '-',
                 'both in WordNet': bool(la and lb),
                 'freq(word)': lemma_freq[la] if la else 0, 'freq(antonym)': lemma_freq[lb] if lb else 0,
                 'similarity': round(score, 3), 'common hypernym': lcs.head_word() if lcs else '-'})
lex_ant = pd.DataFrame(rows)
print(f"Pairs with both words in IndoWordNet: {lex_ant['both in WordNet'].sum()} / {len(lex_ant)}")
lex_ant

In [ ]:
# Semantic profile of sentences: lexical relations of every content word (gold tags from Step 3)
ANTONYMS = defaultdict(set)
for df_ant in (deriv_ant, lex_ant):
    for a, b in zip(df_ant['word'], df_ant['antonym']):
        la, lb = wn_lemma(a) or a, wn_lemma(b) or b
        ANTONYMS[la].add(lb)
        ANTONYMS[lb].add(la)

TAG_TO_WN = {'NOUN': PosTag.NOUN, 'VERB': PosTag.VERB, 'ADJ': PosTag.ADJECTIVE, 'ADV': PosTag.ADVERB}


def semantic_profile(tagged_sent):
    rows = []
    for w, t in tagged_sent:
        if t not in TAG_TO_WN:
            continue
        lemma = wn_lemma(w)
        senses = (iwn.synsets(lemma, pos=TAG_TO_WN[t]) or iwn.synsets(lemma)) if lemma else []
        s = senses[0] if senses else None
        rows.append({
            'word': w, 'tag': t, 'lemma': lemma or '-', 'senses': len(senses),
            'synonyms': ', '.join([n for n in s.lemma_names() if n != lemma][:4]) if s else '',
            'hypernym': ', '.join(head_words(hypernyms(s), 2)) if s else '',
            'antonyms': ', '.join(sorted(ANTONYMS.get(lemma, []))) if lemma else '',
            'gloss': s.gloss()[:60] if s else '',
        })
    return pd.DataFrame(rows)


for i in (1, 13):
    print(' '.join(w for w, _ in gold_sents[i]))
    display(semantic_profile(gold_sents[i]))